In [1]:
import os

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

In [2]:
from pathlib import Path

%cd /content
!git lfs install

if not Path("/content/Project-phase-2").exists():
    !git clone https://github.com/CMU-18734-17731-2026-Fall/Project-phase-2.git

%cd /content/Project-phase-2
!git lfs pull

/content
Git LFS initialized.
/content/Project-phase-2


In [3]:
%pip install -r requirements.txt

In [4]:
%pip uninstall -y torchvision

In [3]:
!apt-get update -qq
!apt-get install -y git-lfs

%cd /content/Project-phase-2
!git lfs install
!git lfs pull

In [4]:
%cd /content/Project-phase-2

import subprocess
import sys

result = subprocess.run(
    [sys.executable, "scripts/verify_release.py"],
    capture_output=True,
    text=True,
)

print("Python:", sys.version)
print("STDOUT:\n", result.stdout)
print("STDERR:\n", result.stderr)
print("Exit code:", result.returncode)

/content/Project-phase-2
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
STDOUT:
 Verified 8 release hashes
Verified data schemas, counts, and public train labels
Verified train, val, and final adapter configurations
Verified that hidden labels and organizer archives are absent
Verified clean notebook metadata and outputs
Verified that stale links and personal paths are absent
Phase 2 release verification: PASS

STDERR:
 
Exit code: 0


In [5]:
import os
import subprocess
import sys

env = os.environ.copy()
env["USE_TF"] = "0"
env["USE_FLAX"] = "0"

result = subprocess.run(
    [
        sys.executable,
        "-c",
        """
from transformers.models.bloom.modeling_bloom import BloomPreTrainedModel
from transformers.models.gpt2.modeling_gpt2 import GPT2LMHeadModel
from peft import PeftModel
print("Imports successful")
""",
    ],
    env=env,
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)

Imports successful




In [6]:
!pip show transformers peft | grep -E "^(Name|Version)"

In [6]:
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from peft import PeftModel
from sklearn.metrics import roc_auc_score, roc_curve
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
PHASE = "train"
BATCH_SIZE = 2
MAX_LENGTH = 512
MODEL_NAME = "gpt2_3_lora32_adamw_b8_lr2"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path("/content/Project-phase-2")
DATA_DIR = ROOT / "data" / PHASE
ADAPTER_DIR = ROOT / "models" / PHASE / MODEL_NAME

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_dtype = torch.float16 if device.type == "cuda" else torch.float32

print("Phase:", PHASE)
print("Device:", device)

Phase: train
Device: cuda


In [7]:
def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


test_items = read_json(DATA_DIR / "test.json")

assert len(test_items) == 2000
assert all(isinstance(text, str) for text in test_items)

adapter_config = read_json(ADAPTER_DIR / "adapter_config.json")
base_model_name = adapter_config["base_model_name_or_path"]

tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_DIR,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Number of texts:", len(test_items))
print("Base model:", base_model_name)

Number of texts: 2000
Base model: gpt2


In [8]:
from pathlib import Path
print(Path("/content/Project-phase-2").exists())

True


In [10]:
for name in ("ROOT", "tokenizer", "test_items", "DRIVE", "SHADOW_MODEL_DIR", "val_B5", "pair_losses"):
    print(f"{name:18s}", name in globals())

ROOT               True
tokenizer          True
test_items         True
DRIVE              False
SHADOW_MODEL_DIR   False
val_B5             False
pair_losses        False


In [11]:
# RESUME — restore from Drive and rebuild B5 state for dev and validation. No GPU work.
from google.colab import drive
import shutil, gc, hashlib, os, subprocess, sys, zipfile, io
from scipy.stats import rankdata
from tqdm.auto import tqdm

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/MIA_phase2")
assert DRIVE.exists(), "Drive folder MIA_phase2 not found"

# ---------- 1. Copy files back (handles the different backup layouts we used) ----------
def find_src(kind, name):
    for p in (DRIVE / kind / name, DRIVE / f"{kind}_{name}", DRIVE / name, DRIVE / f"data_{name}"):
        ok_model = kind != "models" or (p / "adapter_model.safetensors").exists()
        ok_data = kind != "data" or (p / "train_finetune.json").exists()
        if p.exists() and ok_model and ok_data:
            return p
    return None

missing = []
for name in ("shadow_1001", "shadow_1002", "shadow_inA", "shadow_inB"):
    for kind in ("models", "data"):
        src = find_src(kind, name)
        if src is None:
            missing.append(f"{kind}/{name}")
        else:
            shutil.copytree(src, ROOT / kind / name, dirs_exist_ok=True)

if missing:
    print("Not found in the expected Drive locations:", missing)
    print("Search Drive for other copies before concluding they are lost.")
    print("Cached losses cover all dev/val B5 work. Adapters are needed only for new inference;")
    print("shadow_1001 and shadow_1002 adapters ARE needed for the final phase (their final-text losses are not cached).")

halves_record = None
for p in (DRIVE / "data" / "inpair_halves.json", DRIVE / "inpair_halves.json"):
    if p.exists():
        shutil.copy(p, ROOT / "data" / "inpair_halves.json")
        halves_record = read_json(p)
        break
else:
    print("inpair_halves.json not found; IN membership will rely on the shadows' training files only.")

CACHE_DIR = ROOT / "cache"
shutil.copytree(DRIVE / "cache", CACHE_DIR, dirs_exist_ok=True)

# ---------- 2. Texts, labels, constants ----------
val_texts = read_json(ROOT / "data" / "val" / "test.json")
final_texts = read_json(ROOT / "data" / "final" / "test.json")
labels = np.asarray(read_json(ROOT / "data" / "train" / "test_label.json"), dtype=np.int64)
SHADOW_TRAIN_SIZE = len(read_json(ROOT / "data" / "train" / "train_finetune.json"))
CANDIDATE_COUNT = 2 * SHADOW_TRAIN_SIZE
SHADOW_SEED, SEED2 = 1001, 1002
SHADOW_MODEL_DIR = ROOT / "models" / "shadow_1001"
MODEL2 = ROOT / "models" / "shadow_1002"
env = os.environ.copy()
env.update(USE_TF="0", USE_FLAX="0", TOKENIZERS_PARALLELISM="false")

# ---------- 3. Cached losses ----------
def load_split(fname, key="losses"):
    z = np.load(CACHE_DIR / fname)
    return [a.astype(np.float32) for a in np.split(z[key], np.cumsum(z["lengths"])[:-1])]

cached_target_losses = load_split("train_base_target_losses.npz", "target_losses")
cached_base_losses   = load_split("train_base_target_losses.npz", "base_losses")
cached_shadow_losses = load_split("train_shadow_1001_losses.npz", "shadow_losses")
extra = {p: load_split(f"train_ref_{p}_losses.npz") for p in ("val", "final")}
shadow2_dev = load_split("train_shadow_1002_losses.npz")
shadow2_val = load_split("val_shadow_1002_losses.npz")
val_target_losses = load_split("val_shadow_1001_results.npz", "target_losses")
val_shadow_losses = load_split("val_shadow_1001_results.npz", "shadow_losses")
val_refs = {p: load_split(f"val_ref_{p}_losses.npz") for p in ("train", "final")}
pair_losses = {(n, ph): load_split(f"{ph}_shadow_{n}_losses.npz")
               for n in ("inA", "inB") for ph in ("train", "val", "final")}

# Count checks first: zip() would silently drop missing examples.
assert len(cached_target_losses) == len(test_items) == len(labels) == 2000
assert len(val_target_losses) == len(val_texts) == 2000

for arrs in (cached_base_losses, cached_shadow_losses, extra["val"], extra["final"], shadow2_dev,
             pair_losses[("inA", "train")], pair_losses[("inB", "train")]):
    assert len(arrs) == len(cached_target_losses)
    assert all(a.shape == t.shape for a, t in zip(arrs, cached_target_losses))
for arrs in (val_shadow_losses, shadow2_val, val_refs["train"], val_refs["final"],
             pair_losses[("inA", "val")], pair_losses[("inB", "val")]):
    assert len(arrs) == len(val_target_losses)
    assert all(a.shape == t.shape for a, t in zip(arrs, val_target_losses))
for n in ("inA", "inB"):
    assert len(pair_losses[(n, "final")]) == len(final_texts) == 2000

# ---------- 4. Helpers ----------
def extract_text(record):
    return record if isinstance(record, str) else record["text"]

def normalize_text(text):
    return " ".join(text.split())

def prefix_keys(texts):
    keys = []
    for start in range(0, len(texts), 256):
        enc = tokenizer(texts[start:start + 256], add_special_tokens=False,
                        truncation=True, max_length=MAX_LENGTH, padding=False)
        for ids in enc["input_ids"]:
            keys.append(hashlib.sha256(np.asarray(ids, dtype="<i4").tobytes()).hexdigest())
    return keys

@torch.inference_mode()
def collect_shadow_losses(shadow_model, tokenizer, texts, device, batch_size, max_length):
    cached = []
    for start in tqdm(range(0, len(texts), batch_size)):
        batch = tokenizer(texts[start:start + batch_size], return_tensors="pt",
                          padding=True, truncation=True, max_length=max_length)
        batch = {k: v.to(device) for k, v in batch.items()}
        targets = batch["input_ids"][:, 1:]
        mask = batch["attention_mask"][:, 1:].bool()
        logits = shadow_model(**batch, use_cache=False).logits[:, :-1, :]
        losses = F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]),
                                 targets.reshape(-1), reduction="none").reshape(targets.shape)
        for row, m in zip(losses, mask):
            cached.append(row[m].float().cpu().numpy())
    return cached

def load_target(phase_or_path):
    path = phase_or_path if isinstance(phase_or_path, Path) else ROOT / "models" / phase_or_path / MODEL_NAME
    base = AutoModelForCausalLM.from_pretrained(base_model_name, torch_dtype=model_dtype)
    m = PeftModel.from_pretrained(base, path)
    m.config.pad_token_id = tokenizer.pad_token_id
    return m.to(device).eval()

def tpr1(lab, s):
    fpr, tpr, _ = roc_curve(lab, s)
    return np.max(tpr[fpr < 0.01])

def report(name, scores):
    fpr, tpr, _ = roc_curve(labels, scores)
    print(f"{name:32s} | AUC={roc_auc_score(labels, scores):.4f} | "
          f"TPR@FPR<1%={np.max(tpr[fpr < 0.01]):.4f} | TPR@FPR<5%={np.max(tpr[fpr < 0.05]):.4f}")

# Dev split-half evaluation check (seed 0). Unrelated to the IN-pair assignment.
idx = np.random.default_rng(0).permutation(len(test_items))
halves = [idx[: len(idx) // 2], idx[len(idx) // 2 :]]

def score_B(ref_lists, targets):
    out = []
    for i, t in enumerate(targets):
        refs = [r[i] for r in ref_lists if r[i] is not None]
        d = np.mean(np.stack(refs), axis=0) - t
        out.append(d.mean() * np.sqrt(d.size))
    return np.asarray(out, dtype=np.float64)

shadow_texts  = [r["text"] for r in read_json(ROOT / "data" / "shadow_1001" / "train_finetune.json")]
shadow2_texts = [r["text"] for r in read_json(ROOT / "data" / "shadow_1002" / "train_finetune.json")]
evaluation_texts = test_items + val_texts
blocked_texts = {normalize_text(t) for t in evaluation_texts}
blocked_prefixes = set(prefix_keys(evaluation_texts)) | set(prefix_keys(sorted(blocked_texts)))

train_ft = {normalize_text(extract_text(r)) for r in read_json(ROOT / "data" / "train" / "train_finetune.json")}
in_train_ft = np.array([normalize_text(t) in train_ft for t in val_texts])
train_ref_val = [None if in_train_ft[i] else val_refs["train"][i] for i in range(len(val_texts))]

# IN membership: the shadows' actual training files are the source of truth.
PAIR = {n: {"data": ROOT / "data" / f"shadow_{n}", "model": ROOT / "models" / f"shadow_{n}"}
        for n in ("inA", "inB")}
pair_sets = {n: {normalize_text(r["text"]) for r in read_json(c["data"] / "train_finetune.json")}
             for n, c in PAIR.items()}

if halves_record is not None:   # cross-check the saved split against the training files
    eval_sets = {"train": test_items, "val": val_texts, "final": final_texts}
    for phase, texts in eval_sets.items():
        for side, n in ((0, "inA"), (1, "inB")):
            assert all(normalize_text(texts[i]) in pair_sets[n] for i in halves_record[phase][side]), \
                f"split record disagrees with {n} training data ({phase})"
    print("Split record matches IN-shadow training files.")

def online_scores(texts, targets, out_fixed, phase):
    n = len(texts)
    in_flag = {name: np.array([normalize_text(t) in pair_sets[name] for t in texts]) for name in PAIR}
    per_tok = []
    for i in range(n):
        outs = [r[i] for r in out_fixed if r[i] is not None]
        for name in PAIR:
            if not in_flag[name][i]:
                outs.append(pair_losses[(name, phase)][i])
        per_tok.append((np.stack(outs).mean(0), targets[i]))
    return {"B5": np.array([(o - t).mean() * np.sqrt(t.size) for o, t in per_tok])}

# ---------- 5. Verify against known results ----------
dev_out_fixed = [cached_shadow_losses, shadow2_dev, extra["val"], extra["final"]]
B5_dev = online_scores(test_items, cached_target_losses, dev_out_fixed, "train")["B5"]

val_out_fixed = [val_shadow_losses, shadow2_val, val_refs["final"], train_ref_val]
val_B5 = online_scores(val_texts, val_target_losses, val_out_fixed, "val")["B5"]

assert np.isfinite(B5_dev).all()
assert np.isfinite(val_B5).all()
assert np.isclose(tpr1(labels, B5_dev), 0.679, atol=1e-9), f"dev TPR is {tpr1(labels, B5_dev)}"
assert in_train_ft.sum() == 39
report("B5 dev", B5_dev)

# Validation: compare against the predictions actually submitted (scored 0.691 on Codabench).
zip_candidates = [p for p in (DRIVE / "project_phase2_val_submission_B5.zip",
                              ROOT / "project_phase2_val_submission_B5.zip") if p.exists()]
if zip_candidates:
    with zipfile.ZipFile(zip_candidates[0]) as zf:
        submitted = np.loadtxt(io.BytesIO(zf.read("val/prediction.csv")))
    assert submitted.shape == val_B5.shape
    assert np.allclose(submitted, val_B5, rtol=0, atol=1e-9), \
        f"max abs diff {np.abs(submitted - val_B5).max()}"
    print("val_B5 matches the submitted B5 predictions exactly.")
else:
    print("B5 ZIP not found on Drive; validation restoration is unverified.")

print("Resume complete: B5 state restored for dev and validation.")

Mounted at /content/drive
Split record matches IN-shadow training files.
B5 dev                           | AUC=0.9692 | TPR@FPR<1%=0.6790 | TPR@FPR<5%=0.8560
val_B5 matches the submitted B5 predictions exactly.
Resume complete: B5 state restored for dev and validation.


In [12]:
# FINAL-1 — Final-phase losses (target + OUT refs), final B5 scores, final ZIP. Cached per model.
final_losses = {}
specs = {"target": "final", "shadow1": SHADOW_MODEL_DIR, "shadow2": MODEL2,
         "train_tgt": "train", "val_tgt": "val"}

for key, src in specs.items():
    f = CACHE_DIR / f"final_{key}_losses.npz"
    if f.exists():
        final_losses[key] = load_split(f.name)
        continue
    m = load_target(src)
    final_losses[key] = collect_shadow_losses(m, tokenizer, final_texts, device, BATCH_SIZE, MAX_LENGTH)
    m.to("cpu"); del m; gc.collect(); torch.cuda.empty_cache()
    np.savez_compressed(f, lengths=np.asarray([len(x) for x in final_losses[key]]),
                        losses=np.concatenate(final_losses[key]))

tgt_final = final_losses["target"]
assert len(tgt_final) == len(final_texts) == 2000
for arrs in [final_losses[k] for k in specs if k != "target"] + \
            [pair_losses[("inA", "final")], pair_losses[("inB", "final")]]:
    assert len(arrs) == len(tgt_final)
    assert all(a.shape == t.shape for a, t in zip(arrs, tgt_final))

# Train target saw some final texts: drop it as a reference for those.
in_train_ft_final = np.array([normalize_text(t) in train_ft for t in final_texts])
print("Final texts in train target's fine-tuning set:", in_train_ft_final.sum(), "(expected 31)")
train_ref_final = [None if in_train_ft_final[i] else final_losses["train_tgt"][i] for i in range(2000)]

final_out_fixed = [final_losses["shadow1"], final_losses["shadow2"], train_ref_final, final_losses["val_tgt"]]
final_B5 = online_scores(final_texts, tgt_final, final_out_fixed, "final")["B5"]
assert final_B5.shape == (2000,) and np.isfinite(final_B5).all()
print(f"Score spread  val: mean={val_B5.mean():.3f} sd={val_B5.std():.3f} | "
      f"final: mean={final_B5.mean():.3f} sd={final_B5.std():.3f}")

prediction_path = ROOT / "data" / "final" / "prediction.csv"
submission_path = ROOT / "project_phase2_final_submission_B5.zip"
np.savetxt(prediction_path, final_B5, fmt="%.17g")
for cmd in (
    [sys.executable, str(ROOT / "scripts" / "make_submission.py"), "--phase", "final",
     "--prediction", str(prediction_path), "--output", str(submission_path)],
    [sys.executable, str(ROOT / "scripts" / "validate_submission.py"),
     str(submission_path), "--phase", "final"],
):
    r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    print(r.stdout, r.stderr); assert r.returncode == 0

shutil.copytree(CACHE_DIR, DRIVE / "cache", dirs_exist_ok=True)
shutil.copy(submission_path, DRIVE)
print("Final-phase B5 ready and backed up.")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

Final texts in train target's fine-tuning set: 31 (expected 31)
Score spread  val: mean=0.374 sd=0.475 | final: mean=0.357 sd=0.479
Created /content/Project-phase-2/project_phase2_final_submission_B5.zip with 2000 scores
 
Submission validation: PASS (2000 scores)
Score range: [-0.808377, 3.64886]
 
Final-phase B5 ready and backed up.


In [13]:
# ABLATION — cached-only B5 ablations on dev. No training, model loading, or inference.
# Exploratory: dev labels have already been used for tuning.
import pandas as pd

N = len(test_items)
assert N == len(labels) == len(cached_target_losses) == 2000

# ---------- Reference sources on dev (per text; None = excluded for that text) ----------
fixed_sources = {
    "shadow1001":   cached_shadow_losses,
    "shadow1002":   shadow2_dev,
    "val target":   extra["val"],
    "final target": extra["final"],
}
for name, src in fixed_sources.items():
    assert len(src) == N, name
    assert all(s is None or s.shape == t.shape for s, t in zip(src, cached_target_losses)), name

# IN-pair eligibility from actual training-set membership; 0, 1 or 2 eligible models per text.
in_flag_dev = {n: np.array([normalize_text(t) in pair_sets[n] for t in test_items]) for n in PAIR}
pair_out_dev = [[pair_losses[(n, "train")][i] for n in PAIR if not in_flag_dev[n][i]] for i in range(N)]
counts = np.array([len(x) for x in pair_out_dev])
print("Eligible IN-pair OUT references per dev text:",
      {int(k): int((counts == k).sum()) for k in np.unique(counts)})

def ablation_score(alpha=0.5, drop=None):
    out = np.empty(N, dtype=np.float64)
    for i in range(N):
        refs = [src[i] for name, src in fixed_sources.items() if name != drop and src[i] is not None]
        if drop != "IN-pair OUT":
            refs += pair_out_dev[i]
        assert refs, f"text {i} has no eligible references"
        t = cached_target_losses[i]
        d = np.mean(np.stack(refs), axis=0) - t
        out[i] = d.mean() * np.power(float(t.size), alpha)
    return out

# ---------- 1. Reproduce B5 ----------
abl_B5 = ablation_score(0.5)
assert np.allclose(abl_B5, B5_dev, rtol=0, atol=1e-9), f"max diff {np.abs(abl_B5 - B5_dev).max()}"
assert np.isclose(tpr1(labels, abl_B5), 0.679, atol=1e-9)

# ---------- 2–3. Candidates ----------
candidates = {"B5 (alpha=0.5, all refs)": abl_B5}
for a in (0.0, 0.25, 0.75):
    candidates[f"alpha={a}"] = ablation_score(a)
for src in list(fixed_sources) + ["IN-pair OUT"]:
    candidates[f"drop {src}"] = ablation_score(0.5, drop=src)

for name, s in candidates.items():
    assert s.shape == (N,), name
    assert np.isfinite(s).all(), name
    assert roc_auc_score(labels, s) > 0.5, f"{name}: score direction looks inverted"

# ---------- Exact, fast strict-FPR TPR (verified against roc_curve) ----------
def tpr_strict(s, y):
    neg, pos = s[y == 0], s[y == 1]
    max_fp = int(np.ceil(0.01 * neg.size)) - 1          # FP / n_neg < 0.01  ->  FP <= max_fp
    v = np.sort(neg)[::-1][max_fp]                       # (max_fp+1)-th largest non-member score
    return np.mean(pos > v)

for name, s in candidates.items():
    assert np.isclose(tpr_strict(s, labels), tpr1(labels, s), atol=1e-12), name

# ---------- 6. Paired, stratified bootstrap of TPR difference vs B5 ----------
B = 2000
rng = np.random.default_rng(20261016)
pos_idx, neg_idx = np.flatnonzero(labels == 1), np.flatnonzero(labels == 0)
boot_idx = [np.concatenate([rng.choice(pos_idx, pos_idx.size), rng.choice(neg_idx, neg_idx.size)])
            for _ in range(B)]
base_boot = np.array([tpr_strict(abl_B5[ix], labels[ix]) for ix in boot_idx])

rows = []
base_tpr = tpr1(labels, abl_B5)
for name, s in candidates.items():
    tpr = tpr1(labels, s)
    diffs = np.array([tpr_strict(s[ix], labels[ix]) for ix in boot_idx]) - base_boot
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    rows.append({"method": name,
                 "AUC": roc_auc_score(labels, s),
                 "TPR@FPR<1%": tpr,
                 "ΔTPR vs B5": tpr - base_tpr,
                 "boot ΔTPR 95% CI": f"[{lo:+.3f}, {hi:+.3f}]",
                 "P(Δ>0)": np.mean(diffs > 0)})

ablation_table = pd.DataFrame(rows).set_index("method")
pd.set_option("display.width", 160)
print(ablation_table.to_string(float_format=lambda x: f"{x:.4f}"))

ablation_table.to_csv(CACHE_DIR / "dev_b5_ablation.csv")
shutil.copy(CACHE_DIR / "dev_b5_ablation.csv", DRIVE / "cache")

Eligible IN-pair OUT references per dev text: {0: 7, 1: 1993}
                            AUC  TPR@FPR<1%  ΔTPR vs B5  boot ΔTPR 95% CI  P(Δ>0)
method                                                                           
B5 (alpha=0.5, all refs) 0.9692      0.6790      0.0000  [+0.000, +0.000]  0.0000
alpha=0.0                0.9642      0.3590     -0.3200  [-0.425, -0.118]  0.0000
alpha=0.25               0.9694      0.6070     -0.0720  [-0.207, +0.028]  0.0935
alpha=0.75               0.9644      0.6310     -0.0480  [-0.092, +0.031]  0.1075
drop shadow1001          0.9686      0.6690     -0.0100  [-0.070, +0.054]  0.2610
drop shadow1002          0.9673      0.6590     -0.0200  [-0.090, +0.047]  0.3845
drop val target          0.9665      0.6520     -0.0270  [-0.160, +0.038]  0.1105
drop final target        0.9654      0.6360     -0.0430  [-0.088, +0.049]  0.2855
drop IN-pair OUT         0.9675      0.6480     -0.0310  [-0.077, +0.047]  0.2100


'/content/drive/MyDrive/MIA_phase2/cache/dev_b5_ablation.csv'

In [14]:
# IN-FEATURE TEST — do cached IN-model losses add information beyond B5? Dev only, cached only.
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

N = len(test_items)
assert N == len(labels) == len(cached_target_losses) == len(B5_dev) == 2000

# ---------- Actual IN membership per text ----------
in_flag_dev = {n: np.array([normalize_text(t) in pair_sets[n] for t in test_items]) for n in PAIR}
n_in = sum(in_flag_dev[n].astype(int) for n in PAIR)
print("IN models per dev text:", {int(k): int((n_in == k).sum()) for k in np.unique(n_in)})
assert (n_in >= 1).all(), "every dev text should be IN for at least one pair model"
two_in = np.flatnonzero(n_in == 2)

# ---------- IN feature: (mean IN loss - mean target loss) * sqrt(T), averaged over actual IN models ----------
g_in = np.empty(N, dtype=np.float64)
for i in range(N):
    t = cached_target_losses[i]
    ins = [pair_losses[(n, "train")][i] for n in PAIR if in_flag_dev[n][i]]
    for a in ins:
        assert a.shape == t.shape
    l_in = np.mean(np.stack(ins), axis=0)          # per-token mean over IN models; no per-text variance
    g_in[i] = (l_in.mean() - t.mean()) * np.sqrt(t.size)
assert np.isfinite(g_in).all()
print(f"corr(B5, g_IN) = {np.corrcoef(B5_dev, g_in)[0, 1]:.3f}")

# ---------- Folds: stratified, duplicates of the same normalized text kept together ----------
norm = [normalize_text(t) for t in test_items]
group_id = {t: k for k, t in enumerate(dict.fromkeys(norm))}
groups = np.array([group_id[t] for t in norm])
print("Duplicate dev texts (groups with >1 member):", int((np.bincount(groups) > 1).sum()))

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=2026)
fold_of = np.full(N, -1)
for k, (_, te) in enumerate(cv.split(np.zeros(N), labels, groups)):
    fold_of[te] = k
assert (fold_of >= 0).all()
for g in np.unique(groups):
    assert len(set(fold_of[groups == g])) == 1

# ---------- Fixed regularization, chosen in advance ----------
def make_model():
    return make_pipeline(StandardScaler(), LogisticRegression(C=1.0, penalty="l2", max_iter=1000))

X_b5 = B5_dev[:, None]
X_b5g = np.column_stack([B5_dev, g_in])
oof = {"LR(B5)": np.full(N, np.nan), "LR(B5+g_IN)": np.full(N, np.nan)}

rows = []
for k in range(5):
    tr, te = np.flatnonzero(fold_of != k), np.flatnonzero(fold_of == k)
    m1 = make_model().fit(X_b5[tr], labels[tr])
    m2 = make_model().fit(X_b5g[tr], labels[tr])
    oof["LR(B5)"][te] = m1.decision_function(X_b5[te])
    oof["LR(B5+g_IN)"][te] = m2.decision_function(X_b5g[te])

    coef2 = m2[-1].coef_[0]
    row = {"fold": k, "n": te.size, "n_nonmem": int((labels[te] == 0).sum()),
           "coef B5 (std)": coef2[0], "coef g_IN (std)": coef2[1]}
    for name, s in (("B5", B5_dev[te]), ("LR(B5)", oof["LR(B5)"][te]),
                    ("LR(B5+g_IN)", oof["LR(B5+g_IN)"][te])):
        row[f"TPR {name}"] = tpr1(labels[te], s)
        row[f"AUC {name}"] = roc_auc_score(labels[te], s)
    rows.append(row)

    # LR on B5 alone is a monotone transform of B5 within a fold, so its metrics must match B5 exactly.
    assert m1[-1].coef_[0][0] > 0
    assert np.isclose(row["TPR LR(B5)"], row["TPR B5"]) and np.isclose(row["AUC LR(B5)"], row["AUC B5"])

assert all(np.isfinite(v).all() for v in oof.values())

fold_table = pd.DataFrame(rows).set_index("fold")
pd.set_option("display.width", 200)
print("\nPer held-out fold (strict FPR<0.01 allows only ~1 false positive per fold of ~200 non-members):")
print(fold_table.to_string(float_format=lambda x: f"{x:.4f}"))

d = fold_table["TPR LR(B5+g_IN)"] - fold_table["TPR B5"]
print(f"\nΔTPR (B5+g_IN − B5) per fold: {np.round(d.values, 3)} | folds improved: {(d > 0).sum()}/5 "
      f"| mean Δ: {d.mean():+.4f}")

print("\nPooled out-of-fold (note: scores come from 5 different fitted models):")
for name, s in (("B5 (unchanged)", B5_dev), ("LR(B5) OOF", oof["LR(B5)"]),
                ("LR(B5+g_IN) OOF", oof["LR(B5+g_IN)"])):
    print(f"  {name:18s} | AUC={roc_auc_score(labels, s):.4f} | TPR@FPR<1%={tpr1(labels, s):.4f}")

# ---------- Diagnostic: texts trained into both IN models (kept in main evaluation) ----------
print(f"\nTexts IN for both pair models ({len(two_in)}): no IN-pair OUT reference for these.")
pct = lambda s, i: 100 * (s < s[i]).mean()
print(pd.DataFrame({
    "idx": two_in, "label": labels[two_in], "fold": fold_of[two_in],
    "B5": B5_dev[two_in], "B5 pct": [pct(B5_dev, i) for i in two_in],
    "g_IN": g_in[two_in],
    "LR(B5+g) OOF": oof["LR(B5+g_IN)"][two_in],
    "LR(B5+g) pct": [pct(oof["LR(B5+g_IN)"], i) for i in two_in],
}).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# ---------- Save ----------
out = CACHE_DIR / "dev_in_feature_cv.npz"
np.savez(out, fold=fold_of, groups=groups, labels=labels, B5=B5_dev, g_in=g_in, n_in=n_in,
         oof_lr_b5=oof["LR(B5)"], oof_lr_b5_gin=oof["LR(B5+g_IN)"])
fold_table.to_csv(CACHE_DIR / "dev_in_feature_cv_folds.csv")
for f in (out, CACHE_DIR / "dev_in_feature_cv_folds.csv"):
    shutil.copy(f, DRIVE / "cache")
print("\nSaved OOF predictions and fold assignments.")

IN models per dev text: {1: 1993, 2: 7}
corr(B5, g_IN) = 0.793
Duplicate dev texts (groups with >1 member): 0

Per held-out fold (strict FPR<0.01 allows only ~1 false positive per fold of ~200 non-members):
        n  n_nonmem  coef B5 (std)  coef g_IN (std)  TPR B5  AUC B5  TPR LR(B5)  AUC LR(B5)  TPR LR(B5+g_IN)  AUC LR(B5+g_IN)
fold                                                                                                                         
0     400       185         3.6129           1.1220  0.6140  0.9685      0.6140      0.9685           0.5488           0.9727
1     400       208         3.8736           1.1457  0.6302  0.9550      0.6302      0.9550           0.7083           0.9606
2     400       217         3.5016           1.2922  0.6612  0.9725      0.6612      0.9725           0.6940           0.9715
3     400       191         3.6342           1.1018  0.6603  0.9690      0.6603      0.9690           0.7081           0.9731
4     400       199         3.4567   

In [15]:
# HYBRID-VAL — freeze LR(B5 + g_IN), fit on all dev, score validation. Cached only; B5 untouched.
import json, joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Exact settings used in cross-validation (fixed in advance; no tuning here).
HYBRID_SETTINGS = {
    "features": ["B5", "g_IN"],
    "B5": "mean over eligible OUT refs of per-token loss, minus target loss; mean over tokens; × sqrt(T). "
          "Val refs: shadow1001, shadow1002, final target, train target (excluded for 39 texts in its "
          "fine-tuning set), plus whichever IN-pair models did not train on the text.",
    "g_IN": "(mean over IN-pair models that actually trained on the text of per-token loss) − target "
            "per-token loss; mean over tokens; × sqrt(T). No per-text variance.",
    "model": "StandardScaler → LogisticRegression(penalty='l2', C=1.0, max_iter=1000)",
    "fit_data": "all 2,000 dev examples",
    "output": "decision_function (larger = more likely member)",
}

# ---------- 1. Fit on all dev ----------
assert len(B5_dev) == len(g_in) == len(labels) == 2000
assert np.isfinite(B5_dev).all() and np.isfinite(g_in).all()
hybrid = make_pipeline(StandardScaler(), LogisticRegression(penalty="l2", C=1.0, max_iter=1000))
hybrid.fit(np.column_stack([B5_dev, g_in]), labels)

coef = hybrid[-1].coef_[0]
print(f"Standardized coefficients | B5: {coef[0]:+.4f} | g_IN: {coef[1]:+.4f} | "
      f"intercept: {hybrid[-1].intercept_[0]:+.4f}")
print(f"Scaler mean: {hybrid[0].mean_} | scale: {hybrid[0].scale_}")

# ---------- 2. Validation g_IN from actual IN membership ----------
assert len(val_texts) == len(val_target_losses) == len(val_B5) == 2000
in_flag_val = {n: np.array([normalize_text(t) in pair_sets[n] for t in val_texts]) for n in PAIR}
n_in_val = sum(in_flag_val[n].astype(int) for n in PAIR)
print("IN models per validation text:", {int(k): int((n_in_val == k).sum()) for k in np.unique(n_in_val)})
assert (n_in_val >= 1).all(), "every validation text must be IN for at least one pair model"

g_in_val = np.empty(2000, dtype=np.float64)
for i in range(2000):
    t = val_target_losses[i]
    ins = [pair_losses[(n, "val")][i] for n in PAIR if in_flag_val[n][i]]
    for a in ins:
        assert a.shape == t.shape
    g_in_val[i] = (np.mean(np.stack(ins), axis=0).mean() - t.mean()) * np.sqrt(t.size)
assert np.isfinite(g_in_val).all()

# Transfer sanity check: do the features look similar on dev and validation?
for name, dv, vv in (("B5", B5_dev, val_B5), ("g_IN", g_in, g_in_val)):
    print(f"{name:5s} dev mean/sd = {dv.mean():+.3f}/{dv.std():.3f} | val mean/sd = {vv.mean():+.3f}/{vv.std():.3f}")

# ---------- 3. Score validation with the frozen model ----------
val_hybrid = hybrid.decision_function(np.column_stack([val_B5, g_in_val]))

# ---------- 4. Checks and packaging ----------
assert val_hybrid.shape == (2000,) and np.isfinite(val_hybrid).all()
assert len(val_hybrid) == len(val_texts)          # same order as data/val/test.json (val_B5 order)
print(f"corr(val_hybrid, val_B5) = {np.corrcoef(val_hybrid, val_B5)[0, 1]:.4f}")

prediction_path = ROOT / "data" / "val" / "prediction_hybrid.csv"
submission_path = ROOT / "project_phase2_val_submission_B5_gIN.zip"
np.savetxt(prediction_path, val_hybrid, fmt="%.17g")
for cmd in (
    [sys.executable, str(ROOT / "scripts" / "make_submission.py"), "--phase", "val",
     "--prediction", str(prediction_path), "--output", str(submission_path)],
    [sys.executable, str(ROOT / "scripts" / "validate_submission.py"),
     str(submission_path), "--phase", "val"],
):
    r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    print(r.stdout, r.stderr); assert r.returncode == 0

with zipfile.ZipFile(submission_path) as zf:     # round-trip check of the packaged file
    packed = np.loadtxt(io.BytesIO(zf.read("val/prediction.csv")))
assert np.allclose(packed, val_hybrid, rtol=0, atol=1e-9)

# ---------- 5. Save model, settings, features, ZIP to Drive ----------
HYB_DIR = DRIVE / "hybrid_B5_gIN"
HYB_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(hybrid, HYB_DIR / "hybrid_scaler_logreg.joblib")
HYBRID_SETTINGS.update({
    "coef_standardized": coef.tolist(),
    "intercept": float(hybrid[-1].intercept_[0]),
    "scaler_mean": hybrid[0].mean_.tolist(),
    "scaler_scale": hybrid[0].scale_.tolist(),
    "val_in_model_counts": {int(k): int((n_in_val == k).sum()) for k in np.unique(n_in_val)},
    "dev_cv_reference": "5-fold StratifiedGroupKFold(seed 2026): pooled OOF TPR 0.700 vs B5 0.679; "
                        "4/5 folds improved; one fold −6.5 pts.",
})
with open(HYB_DIR / "settings.json", "w") as f:
    json.dump(HYBRID_SETTINGS, f, indent=2)
np.savez(HYB_DIR / "features.npz", dev_B5=B5_dev, dev_g_in=g_in, dev_labels=labels,
         val_B5=val_B5, val_g_in=g_in_val, val_n_in=n_in_val, val_hybrid=val_hybrid)
shutil.copy(submission_path, DRIVE)
shutil.copy(submission_path, HYB_DIR)
print("Saved hybrid model, settings, features and ZIP. B5 and its ZIP are unchanged.")

Standardized coefficients | B5: +3.6488 | g_IN: +1.1603 | intercept: +0.3634
Scaler mean: [ 0.37557582 -0.35319294] | scale: [0.47829629 0.52902633]
IN models per validation text: {1: 1993, 2: 7}
B5    dev mean/sd = +0.376/0.478 | val mean/sd = +0.374/0.475
g_IN  dev mean/sd = -0.353/0.529 | val mean/sd = -0.351/0.530
corr(val_hybrid, val_B5) = 0.9882
Created /content/Project-phase-2/project_phase2_val_submission_B5_gIN.zip with 2000 scores
 
Submission validation: PASS (2000 scores)
Score range: [-12.2336, 17.3846]
 
Saved hybrid model, settings, features and ZIP. B5 and its ZIP are unchanged.


In [16]:
# STAGED-0 — Shared setup for the staged IN/OUT experiment. Run after RESUME (and FINAL-1).
import json, random
import pandas as pd

eval_sets = {"train": test_items, "val": val_texts, "final": final_texts}
PHASES = list(eval_sets)
assert all(len(v) == 2000 for v in eval_sets.values())
eval_items = [(ph, i, t) for ph, texts in eval_sets.items() for i, t in enumerate(texts)]
PH_SLICE = {ph: slice(k * 2000, (k + 1) * 2000) for k, ph in enumerate(PHASES)}

# ---------- Global eval groups: normalized text OR identical scored-token prefix ----------
eval_norm = [normalize_text(t) for _, _, t in eval_items]
eval_pref = prefix_keys([t for _, _, t in eval_items])
parent = list(range(len(eval_items)))
def _find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]
        a = parent[a]
    return a
first_seen = {}
for k, (nk, pk) in enumerate(zip(eval_norm, eval_pref)):
    for key in (("n", nk), ("p", pk)):
        if key in first_seen:
            parent[_find(k)] = _find(first_seen[key])
        else:
            first_seen[key] = k
GROUPS = {}
for k in range(len(eval_items)):
    GROUPS.setdefault(_find(k), []).append(k)
print(f"Eval items: {len(eval_items)} | groups: {len(GROUPS)} | "
      f"multi-item groups: {sum(len(v) > 1 for v in GROUPS.values())}")
EVAL_NORM_SET, EVAL_PREF_SET = set(eval_norm), set(eval_pref)

def membership(train_texts):
    """Actual membership of every eval item: normalized-text or scored-prefix match."""
    ns = {normalize_text(t) for t in train_texts}
    ps = set(prefix_keys(list(train_texts)))
    flags = np.array([(n in ns) or (p in ps) for n, p in zip(eval_norm, eval_pref)])
    return {ph: flags[PH_SLICE[ph]] for ph in PHASES}

# ---------- Auxiliary pool: exclude every eval group (dev, val AND final) ----------
aux_candidates = list(dict.fromkeys(shadow_texts + shadow2_texts))
aux_pref = prefix_keys(aux_candidates)
aux_pool = [t for t, p in zip(aux_candidates, aux_pref)
            if normalize_text(t) not in EVAL_NORM_SET and p not in EVAL_PREF_SET]
print(f"Aux pool after excluding all eval groups: {len(aux_pool)} of {len(aux_candidates)}")

# ---------- Fixed shadows vs final texts (FINAL-1 bug check) ----------
mem_s1, mem_s2 = membership(shadow_texts), membership(shadow2_texts)
for ph in PHASES:
    print(f"{ph:5s} | shadow1001 trained on {mem_s1[ph].sum():3d} | shadow1002 trained on {mem_s2[ph].sum():3d}")
assert not (mem_s1["train"].any() or mem_s1["val"].any() or mem_s2["train"].any() or mem_s2["val"].any())

if (mem_s1["final"].any() or mem_s2["final"].any()) and "final_losses" in globals():
    in_tf_final = np.array([normalize_text(t) in train_ft for t in final_texts])
    fixed_final = [
        [None if mem_s1["final"][i] else final_losses["shadow1"][i] for i in range(2000)],
        [None if mem_s2["final"][i] else final_losses["shadow2"][i] for i in range(2000)],
        [None if in_tf_final[i] else final_losses["train_tgt"][i] for i in range(2000)],
        final_losses["val_tgt"],
    ]
    final_B5_fix = online_scores(final_texts, final_losses["target"], fixed_final, "final")["B5"]
    assert final_B5_fix.shape == (2000,) and np.isfinite(final_B5_fix).all()
    p = ROOT / "data" / "final" / "prediction_B5_fix.csv"
    z = ROOT / "project_phase2_final_submission_B5_fix.zip"
    np.savetxt(p, final_B5_fix, fmt="%.17g")
    for cmd in ([sys.executable, str(ROOT / "scripts" / "make_submission.py"), "--phase", "final",
                 "--prediction", str(p), "--output", str(z)],
                [sys.executable, str(ROOT / "scripts" / "validate_submission.py"), str(z), "--phase", "final"]):
        r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
        print(r.stdout, r.stderr); assert r.returncode == 0
    shutil.copy(z, DRIVE)
    print("Corrected final B5 written as a separate ZIP; the original final ZIP is untouched.")

# ---------- Registry of paired shadows (actual membership + cached losses) ----------
(DRIVE / "models").mkdir(exist_ok=True); (DRIVE / "logs").mkdir(exist_ok=True)
REF_SHAPES = {"train": cached_target_losses, "val": val_target_losses, "final": pair_losses[("inA", "final")]}
MODELS, PAIRS = {}, []

def register_pair(names):
    for n in names:
        tt = [r["text"] for r in read_json(ROOT / "data" / f"shadow_{n}" / "train_finetune.json")]
        need = [ph for ph in PHASES if not (CACHE_DIR / f"{ph}_shadow_{n}_losses.npz").exists()]
        if need:
            m = load_target(ROOT / "models" / f"shadow_{n}")
            for ph in need:
                ls = collect_shadow_losses(m, tokenizer, eval_sets[ph], device, BATCH_SIZE, MAX_LENGTH)
                f = CACHE_DIR / f"{ph}_shadow_{n}_losses.npz"
                np.savez_compressed(f, lengths=np.asarray([len(x) for x in ls]), losses=np.concatenate(ls))
                shutil.copy(f, DRIVE / "cache")
            m.to("cpu"); del m; gc.collect(); torch.cuda.empty_cache()
        loss = {ph: load_split(f"{ph}_shadow_{n}_losses.npz") for ph in PHASES}
        for ph in PHASES:
            assert len(loss[ph]) == 2000
            assert all(a.shape == b.shape for a, b in zip(loss[ph], REF_SHAPES[ph]))
        MODELS[n] = {"mem": membership(tt), "loss": loss}
    if tuple(names) not in PAIRS:
        PAIRS.append(tuple(names))

register_pair(("inA", "inB"))
both = MODELS["inA"]["mem"]["train"] & MODELS["inB"]["mem"]["train"]
print("Existing pair: dev texts IN both models:", int(both.sum()), "(expected 7)")

for k in (2, 3, 4):                     # re-register any new pairs already trained in earlier sessions
    nm = (f"p{k}A", f"p{k}B")
    if all((ROOT / "models" / f"shadow_{n}" / "adapter_model.safetensors").exists() or
           (CACHE_DIR / f"final_shadow_{n}_losses.npz").exists() for n in nm):
        register_pair(nm)
print("Registered pairs:", PAIRS)

def train_cmd(cfg):
    args = [sys.executable, str(ROOT / "ft_llm" / "train_shadow_model.py"),
            "--data-dir", str(cfg["data"]), "--outdir", str(cfg["model"]), "--model-name", base_model_name,
            "--epochs", "3", "--batch-size", "2", "--gradient-accumulation-steps", "4",
            "--block-size", "512", "--lr", "2e-4", "--weight-decay", "0.01", "--seed", str(cfg["seed"]),
            "--lora-r", str(adapter_config["r"]), "--lora-alpha", str(adapter_config["lora_alpha"]),
            "--lora-dropout", str(adapter_config["lora_dropout"]),
            "--target-modules", ",".join(adapter_config["target_modules"]),
            "--precision", "auto", "--gradient-checkpointing"]
    return " ".join(f"'{a}'" for a in args)

Eval items: 6000 | groups: 5981 | multi-item groups: 17
Aux pool after excluding all eval groups: 19948 of 20000
train | shadow1001 trained on   0 | shadow1002 trained on   0
val   | shadow1001 trained on   0 | shadow1002 trained on   0
final | shadow1001 trained on  26 | shadow1002 trained on  26
Created /content/Project-phase-2/project_phase2_final_submission_B5_fix.zip with 2000 scores
 
Submission validation: PASS (2000 scores)
Score range: [-0.75835, 3.64886]
 
Corrected final B5 written as a separate ZIP; the original final ZIP is untouched.
Existing pair: dev texts IN both models: 7 (expected 7)
Registered pairs: [('inA', 'inB')]


In [45]:
# STAGED-3 — Evaluate B_ext, midpoint, and regularized LR at the current number of pairs.
def tm(losses):
    return np.array([a.astype(np.float64).mean() for a in losses])

def tpr_strict(s, y):
    neg, pos = s[y == 0], s[y == 1]
    v = np.sort(neg)[::-1][int(np.ceil(0.01 * neg.size)) - 1]
    return np.mean(pos > v)

def method_scores(mt, L, fixed, P, Pin, kappa, var_fixed=None):
    """mt: target mean loss (N,); L: tokens (N,); fixed: (F,N) OUT means used as references, NaN = excluded;
       P: (K,N) paired-shadow mean losses; Pin: (K,N) actual membership;
       var_fixed: optional (G,N) OUT means used ONLY to estimate pooled variance when pairs can't."""
    N, sq = mt.size, np.sqrt(L)
    Pout = ~Pin
    n_in, n_out = Pin.sum(0), Pout.sum(0)
    F_ok = ~np.isnan(fixed) if fixed.size else np.zeros((0, N), bool)
    n_fix = F_ok.sum(0)
    fix_sum = np.where(F_ok, fixed, 0.0).sum(0)

    cnt = n_fix + n_out
    assert (cnt > 0).all()
    B = ((fix_sum + np.where(Pout, P, 0.0).sum(0)) / cnt - mt) * sq

    assert (n_in > 0).all(), "text with no IN sample"
    no_out = n_out == 0
    assert (~no_out | (n_fix > 0)).all(), "text with no OUT estimate"
    mu_in = np.where(Pin, P, 0.0).sum(0) / n_in
    mu_out = np.where(Pout, P, 0.0).sum(0) / np.maximum(n_out, 1)
    mu_out_f = np.where(no_out, fix_sum / np.maximum(n_fix, 1), mu_out)

    MID = ((mu_out_f + mu_in) / 2 - mt) * sq

    U = P * sq
    def side_var(mask, n, mu):
        ss = np.where(mask, (U - mu * sq) ** 2, 0.0).sum(0)
        df = np.maximum(n - 1, 0)
        var = ss / np.maximum(df, 1)
        pooled = (var * df).sum() / df.sum() if df.sum() > 0 else np.nan
        return var, df, pooled
    v_in, df_in, pool_in = side_var(Pin, n_in, mu_in)
    v_out, df_out, pool_out = side_var(Pout, n_out, mu_out)
    if not np.isfinite(pool_out):
        src = fixed if var_fixed is None else var_fixed
        ok2 = ~np.isnan(src) if src.size else np.zeros((0, N), bool)
        n2 = ok2.sum(0)
        if (n2 >= 2).any():
            Fs = np.where(ok2, src * sq, np.nan)[:, n2 >= 2]
            pool_out = np.nanmean(np.nanvar(Fs, axis=0, ddof=1))
    if not np.isfinite(pool_in):
        pool_in = pool_out
    assert np.isfinite(pool_in) and np.isfinite(pool_out), "cannot estimate pooled variance"
    def shrink(var, df, pool):
        return np.full(var.shape, pool) if np.isinf(kappa) else (var * df + kappa * pool) / (df + kappa)
    n_out_eff = np.where(no_out, n_fix, n_out)
    s_in = shrink(v_in, df_in, pool_in) * (1 + 1 / n_in)
    s_out = np.where(no_out, pool_out, shrink(v_out, df_out, pool_out)) * (1 + 1 / n_out_eff)
    ut, mi, mo = mt * sq, mu_in * sq, mu_out_f * sq
    LR = (-(ut - mi) ** 2 / (2 * s_in) - 0.5 * np.log(s_in)) - (-(ut - mo) ** 2 / (2 * s_out) - 0.5 * np.log(s_out))
    return {"B_ext": B, "midpoint": MID, "LR": LR}, {"no_out_pair": int(no_out.sum())}

KAPPAS = (0.5, 1, 2, 4, 8, 16, np.inf)       # fixed in advance; chosen only on mock targets
names_all = [n for p in PAIRS for n in p]
partner = {a: b for a, b in PAIRS} | {b: a for a, b in PAIRS}
L_all = np.concatenate([[len(a) for a in REF_SHAPES[ph]] for ph in PHASES]).astype(np.float64)
MEANS = {n: np.concatenate([tm(MODELS[n]["loss"][ph]) for ph in PHASES]) for n in names_all}
MEM = {n: np.concatenate([MODELS[n]["mem"][ph] for ph in PHASES]) for n in names_all}

# ---------- Mock targets: each paired shadow as target; partner excluded (its membership is the complement) ----------
# Shadows 1001/1002 as variance-only OUT samples for mock targets (masked where they trained on a text).
def _final_mean(key):
    f = CACHE_DIR / f"final_{key}_losses.npz"
    return tm(load_split(f.name)) if f.exists() else np.full(2000, np.nan)
s1_all = np.concatenate([tm(cached_shadow_losses), tm(val_shadow_losses), _final_mean("shadow1")])
s2_all = np.concatenate([tm(shadow2_dev), tm(shadow2_val), _final_mean("shadow2")])
s1_mem = np.concatenate([mem_s1[ph] for ph in PHASES])
s2_mem = np.concatenate([mem_s2[ph] for ph in PHASES])
VAR_FIXED = np.stack([np.where(s1_mem, np.nan, s1_all), np.where(s2_mem, np.nan, s2_all)])
mock_rows = []
for m in names_all:
    refs = [n for n in names_all if n not in (m, partner[m])]
    if not refs:
        continue
    P = np.stack([MEANS[n] for n in refs]); Pin = np.stack([MEM[n] for n in refs])
    ok = (Pin.sum(0) > 0) & ((~Pin).sum(0) > 0)
    y = MEM[m][ok].astype(int)
    for kappa in KAPPAS:
        sc, _ = method_scores(MEANS[m][ok], L_all[ok], np.empty((0, ok.sum())), P[:, ok], Pin[:, ok],kappa, var_fixed=VAR_FIXED[:, ok])
        for meth, s in sc.items():
            if meth != "LR" and kappa != KAPPAS[0]:
                continue
            mock_rows.append({"mock_target": m, "method": meth if meth != "LR" else f"LR k={kappa}",
                              "n": int(ok.sum()), "TPR": tpr_strict(s, y), "AUC": roc_auc_score(y, s)})

kappa_star, mock_summary = None, None
if mock_rows:
    mock_df = pd.DataFrame(mock_rows)
    mock_summary = mock_df.groupby("method")[["TPR", "AUC"]].agg(["mean", "min", "max"])
    print("Mock-target results (independent of dev labels):")
    print(mock_summary.to_string(float_format=lambda x: f"{x:.4f}"))
    lr = mock_df[mock_df.method.str.startswith("LR")].groupby("method")["TPR"].mean()
    kappa_star = max(KAPPAS, key=lambda k: (round(lr[f"LR k={k}"], 6), k))   # ties -> more shrinkage
    print("Shrinkage chosen on mock targets: kappa* =", kappa_star)
else:
    print("Only one pair registered: no mock targets yet, so LR is not evaluated at this stage.")

# ---------- Dev evaluation ----------
dev_mt = tm(cached_target_losses)
dev_L = np.array([len(a) for a in cached_target_losses], dtype=np.float64)
dev_fixed = np.stack([tm(cached_shadow_losses), tm(shadow2_dev), tm(extra["val"]), tm(extra["final"])])
P_dev = np.stack([tm(MODELS[n]["loss"]["train"]) for n in names_all])
Pin_dev = np.stack([MODELS[n]["mem"]["train"] for n in names_all])
sc_dev, info = method_scores(dev_mt, dev_L, dev_fixed, P_dev, Pin_dev, kappa_star or 4)
print("Dev texts with no OUT paired shadow (midpoint/LR fall back to fixed OUT refs):", info["no_out_pair"])

if PAIRS == [("inA", "inB")]:                 # stage 1: B_ext must reproduce B5
    assert np.isclose(tpr1(labels, sc_dev["B_ext"]), tpr1(labels, B5_dev))
    assert np.allclose(sc_dev["B_ext"], B5_dev, atol=1e-4)

cands = {"B5 (baseline)": B5_dev, "B_ext": sc_dev["B_ext"], "midpoint": sc_dev["midpoint"]}
if kappa_star is not None:
    cands[f"LR (kappa*={kappa_star})"] = sc_dev["LR"]
for k, s in cands.items():
    assert s.shape == (2000,) and np.isfinite(s).all() and roc_auc_score(labels, s) > 0.5, k

# Paired, stratified bootstrap vs B5
B_BOOT = 2000
rng_b = np.random.default_rng(20261017)
pos_idx, neg_idx = np.flatnonzero(labels == 1), np.flatnonzero(labels == 0)
boot = [np.concatenate([rng_b.choice(pos_idx, pos_idx.size), rng_b.choice(neg_idx, neg_idx.size)])
        for _ in range(B_BOOT)]
base_b = np.array([tpr_strict(B5_dev[ix], labels[ix]) for ix in boot])

rows = []
for k, s in cands.items():
    d = np.array([tpr_strict(s[ix], labels[ix]) for ix in boot]) - base_b
    lo, hi = np.percentile(d, [2.5, 97.5])
    mock_tpr = np.nan
    if mock_summary is not None:
        key = k if k in ("B_ext", "midpoint") else (f"LR k={kappa_star}" if k.startswith("LR") else None)
        if key in mock_summary.index:
            mock_tpr = mock_summary.loc[key, ("TPR", "mean")]
    rows.append({"n_pairs": len(PAIRS), "method": k,
                 "dev AUC": roc_auc_score(labels, s), "dev TPR@1%": tpr1(labels, s),
                 "ΔTPR vs B5": tpr1(labels, s) - tpr1(labels, B5_dev),
                 "boot Δ 95% CI": f"[{lo:+.3f}, {hi:+.3f}]", "P(Δ>0)": np.mean(d > 0),
                 "mock TPR (mean)": mock_tpr})

stage = pd.DataFrame(rows)
print(stage.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

csv = DRIVE / "cache" / "staged_results.csv"
hist = pd.read_csv(csv) if csv.exists() else pd.DataFrame()
hist = pd.concat([hist, stage]).drop_duplicates(["n_pairs", "method"], keep="last")
hist.to_csv(csv, index=False)
np.savez(CACHE_DIR / f"staged_dev_scores_{len(PAIRS)}pairs.npz", **{k: v for k, v in sc_dev.items()})
shutil.copy(CACHE_DIR / f"staged_dev_scores_{len(PAIRS)}pairs.npz", DRIVE / "cache")
print("\nResults by number of pairs:")
print(hist.pivot(index="method", columns="n_pairs", values="dev TPR@1%").to_string(float_format=lambda x: f"{x:.4f}"))

Mock-target results (independent of dev labels):
            TPR                  AUC              
           mean    min    max   mean    min    max
method                                            
B_ext    0.6177 0.5501 0.6452 0.9579 0.9541 0.9622
LR k=0.5 0.6651 0.5896 0.7168 0.9662 0.9656 0.9673
LR k=1   0.6651 0.5896 0.7168 0.9662 0.9656 0.9673
LR k=16  0.6651 0.5896 0.7168 0.9662 0.9656 0.9673
LR k=2   0.6651 0.5896 0.7168 0.9662 0.9656 0.9673
LR k=4   0.6651 0.5896 0.7168 0.9662 0.9656 0.9673
LR k=8   0.6651 0.5896 0.7168 0.9662 0.9656 0.9673
LR k=inf 0.6651 0.5896 0.7168 0.9662 0.9656 0.9673
midpoint 0.7092 0.6527 0.7379 0.9754 0.9747 0.9759
Shrinkage chosen on mock targets: kappa* = inf
Dev texts with no OUT paired shadow (midpoint/LR fall back to fixed OUT refs): 0
 n_pairs          method  dev AUC  dev TPR@1%  ΔTPR vs B5    boot Δ 95% CI  P(Δ>0)  mock TPR (mean)
       2   B5 (baseline)   0.9692      0.6790      0.0000 [+0.000, +0.000]  0.0000              NaN
       2   

In [46]:
# STAGED-3b — Unconfounded comparison: every method uses the same OUT references.
def scores_all_out(mt, L, fixed_full, P, Pin, kappa):
    """Treat fixed OUT references as additional OUT samples, so B_ext, midpoint and LR share one OUT set."""
    P_aug = np.vstack([P, fixed_full])
    Pin_aug = np.vstack([Pin, np.zeros(fixed_full.shape, dtype=bool)])
    return method_scores(mt, L, np.empty((0, mt.size)), P_aug, Pin_aug, kappa)

# ---------- Mock targets: pair refs (partner excluded) + shadows 1001/1002 as OUT refs ----------
mock_rows_b = []
for m in names_all:
    refs = [n for n in names_all if n not in (m, partner[m])]
    if not refs:
        continue
    P = np.stack([MEANS[n] for n in refs]); Pin = np.stack([MEM[n] for n in refs])
    ok = (Pin.sum(0) > 0) & ~np.isnan(VAR_FIXED).any(0)        # needs an IN sample; drop shadow-overlap texts
    y = MEM[m][ok].astype(int)
    for kappa in KAPPAS:
        sc, _ = scores_all_out(MEANS[m][ok], L_all[ok], VAR_FIXED[:, ok], P[:, ok], Pin[:, ok], kappa)
        for meth, s in sc.items():
            if meth != "LR" and kappa != KAPPAS[0]:
                continue
            mock_rows_b.append({"mock_target": m, "method": meth if meth != "LR" else f"LR k={kappa}",
                                "n": int(ok.sum()), "TPR": tpr_strict(s, y), "AUC": roc_auc_score(y, s)})

mock_b = pd.DataFrame(mock_rows_b)
mock_b_summary = mock_b.groupby("method")[["TPR", "AUC"]].agg(["mean", "min", "max"])
print("Mock targets, shared OUT set (pair OUT + shadows 1001/1002):")
print(mock_b_summary.to_string(float_format=lambda x: f"{x:.4f}"))
lr_b = mock_b[mock_b.method.str.startswith("LR")].groupby("method")["TPR"].mean()
kappa_b = max(KAPPAS, key=lambda k: (round(lr_b[f"LR k={k}"], 6), k))
print("kappa* (shared OUT set, chosen on mock targets) =", kappa_b)

# ---------- Dev: all 6 OUT references for every method ----------
sc_b, _ = scores_all_out(dev_mt, dev_L, dev_fixed, P_dev, Pin_dev, kappa_b)
assert np.allclose(sc_b["B_ext"], sc_dev["B_ext"], atol=1e-9)    # B_ext must be unchanged
cands_b = {"B5 (baseline)": B5_dev, "B_ext": sc_b["B_ext"],
           "midpoint (all OUT)": sc_b["midpoint"], f"LR (all OUT, kappa*={kappa_b})": sc_b["LR"]}

rows_b = []
for k, s in cands_b.items():
    assert s.shape == (2000,) and np.isfinite(s).all() and roc_auc_score(labels, s) > 0.5, k
    d = np.array([tpr_strict(s[ix], labels[ix]) for ix in boot]) - base_b
    lo, hi = np.percentile(d, [2.5, 97.5])
    key = {"B_ext": "B_ext", "midpoint (all OUT)": "midpoint"}.get(k, f"LR k={kappa_b}" if k.startswith("LR") else None)
    rows_b.append({"n_pairs": len(PAIRS), "method": k,
                   "dev AUC": roc_auc_score(labels, s), "dev TPR@1%": tpr1(labels, s),
                   "ΔTPR vs B5": tpr1(labels, s) - tpr1(labels, B5_dev),
                   "boot Δ 95% CI": f"[{lo:+.3f}, {hi:+.3f}]", "P(Δ>0)": np.mean(d > 0),
                   "mock TPR (mean)": mock_b_summary.loc[key, ("TPR", "mean")] if key in mock_b_summary.index else np.nan})
stage_b = pd.DataFrame(rows_b)
print(stage_b.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

hist = pd.read_csv(csv) if csv.exists() else pd.DataFrame()
hist = pd.concat([hist, stage_b]).drop_duplicates(["n_pairs", "method"], keep="last")
hist.to_csv(csv, index=False)
np.savez(CACHE_DIR / f"staged_dev_scores_allout_{len(PAIRS)}pairs.npz", **sc_b)
shutil.copy(CACHE_DIR / f"staged_dev_scores_allout_{len(PAIRS)}pairs.npz", DRIVE / "cache")

Mock targets, shared OUT set (pair OUT + shadows 1001/1002):
            TPR                  AUC              
           mean    min    max   mean    min    max
method                                            
B_ext    0.7578 0.6874 0.7887 0.9770 0.9757 0.9786
LR k=0.5 0.6717 0.6108 0.7061 0.9724 0.9707 0.9740
LR k=1   0.7489 0.7058 0.7723 0.9746 0.9728 0.9758
LR k=16  0.7875 0.7364 0.8135 0.9762 0.9750 0.9783
LR k=2   0.7802 0.7398 0.8192 0.9755 0.9740 0.9770
LR k=4   0.7874 0.7340 0.8219 0.9759 0.9746 0.9777
LR k=8   0.7949 0.7589 0.8162 0.9761 0.9749 0.9781
LR k=inf 0.7772 0.7069 0.8089 0.9759 0.9746 0.9782
midpoint 0.7383 0.6921 0.7720 0.9790 0.9775 0.9799
kappa* (shared OUT set, chosen on mock targets) = 8
 n_pairs                 method  dev AUC  dev TPR@1%  ΔTPR vs B5    boot Δ 95% CI  P(Δ>0)  mock TPR (mean)
       2          B5 (baseline)   0.9692      0.6790      0.0000 [+0.000, +0.000]  0.0000              NaN
       2                  B_ext   0.9692      0.6950      0.0

'/content/drive/MyDrive/MIA_phase2/cache/staged_dev_scores_allout_2pairs.npz'

In [48]:
# DIAG — Is the real target systematically offset from the shadows? Low quantiles are dominated by non-members.
def out_mean(fixed, P, Pin):
    F_ok, Pout = ~np.isnan(fixed), ~Pin
    s = np.where(F_ok, fixed, 0.0).sum(0) + np.where(Pout, P, 0.0).sum(0)
    return s / (F_ok.sum(0) + Pout.sum(0))

rows = []
g = out_mean(dev_fixed, P_dev, Pin_dev) - dev_mt                         # >0: target fits text better than OUT refs
rows.append({"target": "dev (real)", **dict(zip(["p5", "p10", "p25", "p50"], np.percentile(g, [5, 10, 25, 50]))),
             "nonmember mean": g[labels == 0].mean()})

val_fixed = np.stack([tm(val_shadow_losses), tm(shadow2_val), tm(val_refs["final"]),
                      np.where(in_train_ft, np.nan, tm(val_refs["train"]))])
P_val = np.stack([tm(MODELS[n]["loss"]["val"]) for n in names_all])
Pin_val = np.stack([MODELS[n]["mem"]["val"] for n in names_all])
g = out_mean(val_fixed, P_val, Pin_val) - tm(val_target_losses)
rows.append({"target": "val (real)", **dict(zip(["p5", "p10", "p25", "p50"], np.percentile(g, [5, 10, 25, 50]))),
             "nonmember mean": np.nan})

for m in names_all:
    refs = [n for n in names_all if n not in (m, partner[m])]
    P = np.stack([MEANS[n] for n in refs]); Pin = np.stack([MEM[n] for n in refs])
    ok = ((~Pin).sum(0) > 0) & ~np.isnan(VAR_FIXED).any(0)
    g = out_mean(VAR_FIXED[:, ok], P[:, ok], Pin[:, ok]) - MEANS[m][ok]
    y = MEM[m][ok]
    rows.append({"target": f"mock {m}", **dict(zip(["p5", "p10", "p25", "p50"], np.percentile(g, [5, 10, 25, 50]))),
                 "nonmember mean": g[~y].mean()})

print(pd.DataFrame(rows).set_index("target").to_string(float_format=lambda x: f"{x:+.4f}"))

                p5     p10     p25     p50  nonmember mean
target                                                    
dev (real) -0.0265 -0.0159 -0.0011 +0.0292         +0.0003
val (real) -0.0250 -0.0170 -0.0005 +0.0283             NaN
mock inA   -0.0237 -0.0148 -0.0006 +0.0260         -0.0007
mock inB   -0.0236 -0.0134 +0.0004 +0.0284         +0.0005
mock p2A   -0.0236 -0.0150 -0.0004 +0.0275         -0.0002
mock p2B   -0.0235 -0.0154 +0.0001 +0.0303         +0.0005


In [41]:
# STAGED-1 — Build one new complementary pair; train in the background with per-model backups.
PAIR_NO = 3
names = (f"p{PAIR_NO}A", f"p{PAIR_NO}B")
assert names not in PAIRS and not any((ROOT / "models" / f"shadow_{n}").exists() for n in names), \
    "this pair already exists; use the next PAIR_NO"

rng = np.random.default_rng(7000 + PAIR_NO)          # independent group assignment per pair
side_items, side_n = ([], []), [0, 0]
for g in rng.permutation(sorted(GROUPS)):
    g = int(g)
    s = int(rng.integers(2)) if side_n[0] == side_n[1] else int(side_n[1] < side_n[0])
    side_items[s].extend(GROUPS[g]); side_n[s] += len(GROUPS[g])
print("Eval items per side:", side_n)

new_mem, cfgs = {}, []
for s, name in enumerate(names):
    ev = list(dict.fromkeys(eval_items[k][2] for k in side_items[s]))
    n_aux = SHADOW_TRAIN_SIZE - len(ev)
    assert n_aux > 0
    aux = random.Random(8000 + 10 * PAIR_NO + s).sample(aux_pool, n_aux)
    tt = ev + aux
    random.Random(9000 + 10 * PAIR_NO + s).shuffle(tt)
    assert len(tt) == SHADOW_TRAIN_SIZE
    d = ROOT / "data" / f"shadow_{name}"
    d.mkdir(parents=True, exist_ok=True)
    with (d / "train_finetune.json").open("w", encoding="utf-8") as f:
        json.dump([{"text": t} for t in tt], f, ensure_ascii=False)
    new_mem[name] = membership(tt)
    cfgs.append({"data": d, "model": ROOT / "models" / f"shadow_{name}", "seed": 5000 + 10 * PAIR_NO + s})

for ph in PHASES:                                     # every eval prefix IN exactly one model of this pair
    a, b = new_mem[names[0]][ph], new_mem[names[1]][ph]
    assert (a ^ b).all(), f"{ph}: an eval item is not IN exactly one model of {names}"

side = np.zeros(len(eval_items), dtype=int); side[side_items[1]] = 1
assign = {"pair_no": PAIR_NO, "names": names, "group_seed": 7000 + PAIR_NO,
          "aux_seeds": [8000 + 10 * PAIR_NO + s for s in (0, 1)],
          "shuffle_seeds": [9000 + 10 * PAIR_NO + s for s in (0, 1)],
          "train_seeds": [c["seed"] for c in cfgs],
          "side_by_phase": {ph: side[PH_SLICE[ph]].tolist() for ph in PHASES}}
with open(ROOT / "data" / f"pair{PAIR_NO}_assignment.json", "w") as f:
    json.dump(assign, f)
for n in names:
    shutil.copytree(ROOT / "data" / f"shadow_{n}", DRIVE / "data" / f"shadow_{n}", dirs_exist_ok=True)
shutil.copy(ROOT / "data" / f"pair{PAIR_NO}_assignment.json", DRIVE / "data")

steps = []
for n, cfg in zip(names, cfgs):
    log = ROOT / f"train_shadow_{n}.log"
    steps.append(f"{train_cmd(cfg)} > '{log}' 2>&1")
    steps.append(f"cp -r '{cfg['model']}' '{DRIVE}/models/' && cp '{log}' '{DRIVE}/logs/'")
gc.collect(); torch.cuda.empty_cache()
staged_proc = subprocess.Popen(" && ".join(steps), shell=True, cwd=ROOT, env=env,
                               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print(f"Pair {PAIR_NO} training started (PID {staged_proc.pid}); ~50 min. "
      f"Progress: !tail -2 {ROOT}/train_shadow_{names[0]}.log")

Eval items per side: [3000, 3000]
Pair 3 training started (PID 41933); ~50 min. Progress: !tail -2 /content/Project-phase-2/train_shadow_p3A.log


In [54]:
!tail -2 /content/Project-phase-2/train_shadow_p3B.log

{'train_runtime': 1516.4875, 'train_samples_per_second': 19.783, 'train_steps_per_second': 2.473, 'train_loss': 3.448900537109375, 'epoch': 3.0}
Saved model to /content/Project-phase-2/models/shadow_p3B


In [53]:
# STAGED-1F — Pair 4 with independently sampled FRESH auxiliary data. Build now (CPU); launch after pair 3 is scored.
# Same evaluation assignment procedure, training-set size, configuration and seeds as the planned pair 4.
PAIR_NO = 4
names = (f"p{PAIR_NO}A", f"p{PAIR_NO}B")
assert names not in PAIRS and not any((ROOT / "models" / f"shadow_{n}").exists() for n in names)

rng = np.random.default_rng(7000 + PAIR_NO)                      # planned group seed, unchanged
side_items, side_n = ([], []), [0, 0]
for g in rng.permutation(sorted(GROUPS)):
    g = int(g)
    s = int(rng.integers(2)) if side_n[0] == side_n[1] else int(side_n[1] < side_n[0])
    side_items[s].extend(GROUPS[g]); side_n[s] += len(GROUPS[g])
print("Eval items per side:", side_n)

FRESH_GEN_SEEDS = (44001, 44002)                                 # one independent WikiText draw per model
fresh_pool, fresh_stats = {}, {}
for s, name in enumerate(names):
    cand_dir = ROOT / "data" / f"fresh_candidates_{name}"
    r = subprocess.run([sys.executable, str(ROOT / "data" / "prepare_finetuning_data.py"),
                        "--seed", str(FRESH_GEN_SEEDS[s]), "--num-samples", "12000", "--min-tokens", "25",
                        "--model-name", base_model_name, "--output-dir", str(cand_dir)],
                       env=env, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:]); raise RuntimeError("fresh candidate generation failed")
    cands = list(dict.fromkeys(extract_text(x) for x in read_json(cand_dir / "train_finetune.json")))
    pk = prefix_keys(cands)
    clean = [t for t, p in zip(cands, pk) if normalize_text(t) not in EVAL_NORM_SET and p not in EVAL_PREF_SET]
    fresh_pool[name] = clean
    fresh_stats[name] = {"generated": len(cands), "after_eval_exclusion": len(clean)}
print("Fresh candidate pools:", fresh_stats)

new_mem, cfgs = {}, []
for s, name in enumerate(names):
    ev = list(dict.fromkeys(eval_items[k][2] for k in side_items[s]))
    n_aux = SHADOW_TRAIN_SIZE - len(ev)
    assert len(fresh_pool[name]) >= n_aux, f"{name}: need {n_aux}, have {len(fresh_pool[name])}; raise --num-samples"
    aux = random.Random(8000 + 10 * PAIR_NO + s).sample(fresh_pool[name], n_aux)   # planned sampling seed
    tt = ev + aux
    random.Random(9000 + 10 * PAIR_NO + s).shuffle(tt)                            # planned shuffle seed
    assert len(tt) == SHADOW_TRAIN_SIZE
    d = ROOT / "data" / f"shadow_{name}"
    d.mkdir(parents=True, exist_ok=True)
    with (d / "train_finetune.json").open("w", encoding="utf-8") as f:
        json.dump([{"text": t} for t in tt], f, ensure_ascii=False)
    new_mem[name] = membership(tt)
    cfgs.append({"data": d, "model": ROOT / "models" / f"shadow_{name}", "seed": 5000 + 10 * PAIR_NO + s})
for ph in PHASES:
    a, b = new_mem[names[0]][ph], new_mem[names[1]][ph]
    assert (a ^ b).all(), f"{ph}: an eval item is not IN exactly one model of {names}"

# Measure (do not force) overlap with every existing training set
def aux_set(path):
    return {normalize_text(r["text"] if isinstance(r, dict) else r) for r in read_json(path)} - EVAL_NORM_SET
existing = {"shadow1001": aux_set(ROOT / "data" / "shadow_1001" / "train_finetune.json"),
            "shadow1002": aux_set(ROOT / "data" / "shadow_1002" / "train_finetune.json"),
            "TRAIN TARGET (real)": aux_set(ROOT / "data" / "train" / "train_finetune.json")}
for p in PAIRS:
    for n in p:
        existing[n] = aux_set(ROOT / "data" / f"shadow_{n}" / "train_finetune.json")
new_sets = {n: aux_set(ROOT / "data" / f"shadow_{n}" / "train_finetune.json") for n in names}
ov4 = pd.DataFrame({n: {k: len(new_sets[n] & v) for k, v in {**existing, **new_sets}.items()} for n in names})
print("\nShared auxiliary texts, fresh pair-4 models vs all training sets:")
print(ov4.to_string())

side = np.zeros(len(eval_items), dtype=int); side[side_items[1]] = 1
assign = {"pair_no": PAIR_NO, "names": names, "group_seed": 7000 + PAIR_NO,
          "aux_sampling_seeds": [8000 + 10 * PAIR_NO + s for s in (0, 1)],
          "shuffle_seeds": [9000 + 10 * PAIR_NO + s for s in (0, 1)],
          "train_seeds": [c["seed"] for c in cfgs],
          "aux_source": "FRESH: independent WikiText-103 draws via prepare_finetuning_data.py",
          "fresh_generation_seeds": list(FRESH_GEN_SEEDS), "fresh_pool_stats": fresh_stats,
          "construction_note": "Only the auxiliary source differs from pairs 1-3 (which sampled from the "
                               "shadow1001 ∪ shadow1002 pool). Exploratory single-pair comparison, not a clean "
                               "causal experiment.",
          "overlap_with_existing": ov4.to_dict(),
          "side_by_phase": {ph: side[PH_SLICE[ph]].tolist() for ph in PHASES}}
with open(ROOT / "data" / f"pair{PAIR_NO}_assignment.json", "w") as f:
    json.dump(assign, f, indent=1)
for n in names:
    shutil.copytree(ROOT / "data" / f"shadow_{n}", DRIVE / "data" / f"shadow_{n}", dirs_exist_ok=True)
shutil.copy(ROOT / "data" / f"pair{PAIR_NO}_assignment.json", DRIVE / "data")
print("Pair-4 datasets and construction record saved. Launch after pair 3 is scored.")

Eval items per side: [3000, 3000]
Fresh candidate pools: {'p4A': {'generated': 11997, 'after_eval_exclusion': 11902}, 'p4B': {'generated': 11998, 'after_eval_exclusion': 11909}}

Shared auxiliary texts, fresh pair-4 models vs all training sets:
                      p4A   p4B
shadow1001            106    84
shadow1002             88   100
TRAIN TARGET (real)    88    88
inA                    68    75
inB                    72    63
p2A                    69    67
p2B                    64    72
p4A                  7002    65
p4B                    65  7017
Pair-4 datasets and construction record saved. Launch after pair 3 is scored.


In [57]:
# STAGED-1F-launch — Start pair-4 training only after pair 3 is scored (one training job on the GPU at a time).
assert ("p3A", "p3B") in PAIRS, "run STAGED-2 for pair 3 first"
steps = []
for n, cfg in zip(names, cfgs):
    log = ROOT / f"train_shadow_{n}.log"
    steps.append(f"{train_cmd(cfg)} > '{log}' 2>&1")
    steps.append(f"cp -r '{cfg['model']}' '{DRIVE}/models/' && cp '{log}' '{DRIVE}/logs/'")
gc.collect(); torch.cuda.empty_cache()
staged_proc = subprocess.Popen(" && ".join(steps), shell=True, cwd=ROOT, env=env,
                               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print(f"Pair 4 (fresh aux) training started (PID {staged_proc.pid}); ~50 min.")

Pair 4 (fresh aux) training started (PID 58477); ~50 min.


In [65]:
for n in ("p4A", "p4B"):
    log = ROOT / f"train_shadow_{n}.log"
    print(f"--- {n}")
    if log.exists():
        !tail -2 {log}
    else:
        print("not started yet")
print("chain status:", staged_proc.poll(), "(None = still running, 0 = done, other = failed)")
print("p4A saved:", (ROOT / "models" / "shadow_p4A" / "adapter_model.safetensors").exists(),
      "| p4B saved:", (ROOT / "models" / "shadow_p4B" / "adapter_model.safetensors").exists())

--- p4A
{'train_runtime': 1490.5164, 'train_samples_per_second': 20.127, 'train_steps_per_second': 2.516, 'train_loss': 3.4530891723632813, 'epoch': 3.0}
Saved model to /content/Project-phase-2/models/shadow_p4A
--- p4B
{'train_runtime': 1481.9029, 'train_samples_per_second': 20.244, 'train_steps_per_second': 2.531, 'train_loss': 3.4481060750325523, 'epoch': 3.0}
Saved model to /content/Project-phase-2/models/shadow_p4B
chain status: 0 (None = still running, 0 = done, other = failed)
p4A saved: True | p4B saved: True


In [69]:
pair4 = ("p4A", "p4B")
assert staged_proc.poll() == 0
register_pair(pair4)
for ph in PHASES:
    assert (MODELS["p4A"]["mem"][ph] ^ MODELS["p4B"]["mem"][ph]).all()
print("Registered:", PAIRS)

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

Registered: [('inA', 'inB'), ('p2A', 'p2B'), ('p3A', 'p3B'), ('p4A', 'p4B')]


In [56]:
# STAGED-2 for pair 3 (explicit names; `names` now refers to pair 4 for the launch cell).
pair3 = ("p3A", "p3B")
assert staged_proc.poll() == 0, f"pair 3 not finished/failed: {staged_proc.poll()}"
assert all((ROOT / "models" / f"shadow_{n}" / "adapter_model.safetensors").exists() for n in pair3)
register_pair(pair3)
for ph in PHASES:
    assert (MODELS["p3A"]["mem"][ph] ^ MODELS["p3B"]["mem"][ph]).all()
print("Registered:", PAIRS)

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

Registered: [('inA', 'inB'), ('p2A', 'p2B'), ('p3A', 'p3B')]


In [70]:
# STAGED-4 — Stage evaluation + diagnostics. Formulas unchanged; no variance corrections; validation frozen.

# ---------- 0. Completed, scored pairs only ----------
def pair_complete(pair):
    return all(n in MODELS and all((CACHE_DIR / f"{ph}_shadow_{n}_losses.npz").exists() for ph in PHASES)
               for n in pair)
DONE_PAIRS = [p for p in PAIRS if pair_complete(p)]
skipped = [p for p in PAIRS if p not in DONE_PAIRS]
if skipped:
    print("Skipping incomplete pairs:", skipped)
names_all = [n for p in DONE_PAIRS for n in p]
partner = {a: b for a, b in DONE_PAIRS} | {b: a for a, b in DONE_PAIRS}
N_PAIRS = len(DONE_PAIRS)

MEANS = {n: np.concatenate([tm(MODELS[n]["loss"][ph]) for ph in PHASES]) for n in names_all}
MEM = {n: np.concatenate([MODELS[n]["mem"][ph] for ph in PHASES]) for n in names_all}
L_all = np.concatenate([[len(a) for a in REF_SHAPES[ph]] for ph in PHASES]).astype(np.float64)

def _final_mean(key):
    f = CACHE_DIR / f"final_{key}_losses.npz"
    return tm(load_split(f.name)) if f.exists() else np.full(2000, np.nan)
s1_all = np.concatenate([tm(cached_shadow_losses), tm(val_shadow_losses), _final_mean("shadow1")])
s2_all = np.concatenate([tm(shadow2_dev), tm(shadow2_val), _final_mean("shadow2")])
VAR_FIXED = np.stack([np.where(np.concatenate([mem_s1[ph] for ph in PHASES]), np.nan, s1_all),
                      np.where(np.concatenate([mem_s2[ph] for ph in PHASES]), np.nan, s2_all)])

P_dev = np.stack([tm(MODELS[n]["loss"]["train"]) for n in names_all])
Pin_dev = np.stack([MODELS[n]["mem"]["train"] for n in names_all])

def scores_all_out(mt, L, fixed_full, P, Pin, kappa):
    """Fixed OUT references become extra OUT rows. method_scores has no missing-value mask,
       so every passed reference value must be finite."""
    assert np.isfinite(fixed_full).all(), "missing fixed references must be removed before scoring"
    P_aug = np.vstack([P, fixed_full])
    Pin_aug = np.vstack([Pin, np.zeros(fixed_full.shape, dtype=bool)])
    return method_scores(mt, L, np.empty((0, mt.size)), P_aug, Pin_aug, kappa)

# ---------- 1. Mock targets (shared OUT set; partner always excluded) ----------
mock_rows, partner_log = [], []
for m in names_all:
    refs = [n for n in names_all if n not in (m, partner[m])]
    assert m not in refs and partner[m] not in refs
    partner_log.append({"mock_target": m, "excluded_partner": partner[m], "n_pair_refs": len(refs)})
    if not refs:
        continue
    P = np.stack([MEANS[n] for n in refs]); Pin = np.stack([MEM[n] for n in refs])
    ok = (Pin.sum(0) > 0) & ~np.isnan(VAR_FIXED).any(0)
    y = MEM[m][ok].astype(int)
    for kappa in KAPPAS:
        sc, _ = scores_all_out(MEANS[m][ok], L_all[ok], VAR_FIXED[:, ok], P[:, ok], Pin[:, ok], kappa)
        for meth, s in sc.items():
            if meth != "LR" and kappa != KAPPAS[0]:
                continue
            mock_rows.append({"mock_target": m, "method": meth if meth != "LR" else f"LR k={kappa}",
                              "n": int(ok.sum()), "TPR": tpr_strict(s, y), "AUC": roc_auc_score(y, s)})
mock_df = pd.DataFrame(mock_rows)
mock_sum = mock_df.groupby("method")[["TPR", "AUC"]].agg(["mean", "min", "max"])
lr_m = mock_df[mock_df.method.str.startswith("LR")].groupby("method")["TPR"].mean()
kappa_sel = max(KAPPAS, key=lambda k: (round(lr_m[f"LR k={k}"], 6), k))
print(f"=== Stage: {N_PAIRS} completed pairs: {DONE_PAIRS} ===")
print(pd.DataFrame(partner_log).to_string(index=False))
print("\nMock targets (shared OUT set):")
print(mock_sum.to_string(float_format=lambda x: f"{x:.4f}"))
print("Mock-selected kappa:", kappa_sel)

# ---------- 2. Dev candidates; bootstrap vs B5 and vs B_ext ----------
assert np.isfinite(dev_fixed).all()
sc8, _ = scores_all_out(dev_mt, dev_L, dev_fixed, P_dev, Pin_dev, 8)
scS, _ = scores_all_out(dev_mt, dev_L, dev_fixed, P_dev, Pin_dev, kappa_sel)
cands = {"B5": B5_dev, "B_ext": sc8["B_ext"], "midpoint (all OUT)": sc8["midpoint"],
         "LR (all OUT, κ=8 fixed)": sc8["LR"], f"LR (all OUT, κ*={kappa_sel} mock)": scS["LR"]}
for k, s in cands.items():
    assert s.shape == (2000,) and np.isfinite(s).all() and roc_auc_score(labels, s) > 0.5, k

rng_b = np.random.default_rng(20261017)
pos_idx, neg_idx = np.flatnonzero(labels == 1), np.flatnonzero(labels == 0)
boot = [np.concatenate([rng_b.choice(pos_idx, pos_idx.size), rng_b.choice(neg_idx, neg_idx.size)])
        for _ in range(2000)]
bt = {k: np.array([tpr_strict(s[ix], labels[ix]) for ix in boot]) for k, s in cands.items()}
mock_key = {"B_ext": "B_ext", "midpoint (all OUT)": "midpoint", "LR (all OUT, κ=8 fixed)": "LR k=8",
            f"LR (all OUT, κ*={kappa_sel} mock)": f"LR k={kappa_sel}"}

rows = []
for k, s in cands.items():
    r = {"n_pairs": N_PAIRS, "method": k, "dev AUC": roc_auc_score(labels, s), "dev TPR@1%": tpr1(labels, s)}
    for ref in ("B5", "B_ext"):
        d = bt[k] - bt[ref]
        lo, hi = np.percentile(d, [2.5, 97.5])
        r[f"Δ vs {ref}"] = tpr1(labels, s) - tpr1(labels, cands[ref])
        r[f"95% CI vs {ref}"] = f"[{lo:+.3f}, {hi:+.3f}]"
        r[f"bootstrap fraction Δ>0 vs {ref}"] = np.mean(d > 0)
    mk = mock_key.get(k)
    r["mock TPR mean"] = mock_sum.loc[mk, ("TPR", "mean")] if mk in mock_sum.index else np.nan
    rows.append(r)
stage4 = pd.DataFrame(rows)
print("\nDev stage table:")
print(stage4.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

csv4 = DRIVE / "cache" / "staged4_results.csv"
hist4 = pd.read_csv(csv4) if csv4.exists() else pd.DataFrame()
hist4 = pd.concat([hist4, stage4]).drop_duplicates(["n_pairs", "method"], keep="last")
hist4.to_csv(csv4, index=False)
print("\nDev TPR@1% by number of completed pairs:")
print(hist4.pivot(index="method", columns="n_pairs", values="dev TPR@1%").to_string(float_format=lambda x: f"{x:.4f}"))

# ---------- 3. Auxiliary training-set overlap (eval texts removed). Shows shared data, not causation. ----------
def aux_set(path):
    return {normalize_text(r["text"] if isinstance(r, dict) else r) for r in read_json(path)} - EVAL_NORM_SET
train_sets = {"shadow1001": aux_set(ROOT / "data" / "shadow_1001" / "train_finetune.json"),
              "shadow1002": aux_set(ROOT / "data" / "shadow_1002" / "train_finetune.json")}
for n in names_all:
    train_sets[n] = aux_set(ROOT / "data" / f"shadow_{n}" / "train_finetune.json")
train_sets["TRAIN TARGET (real)"] = aux_set(ROOT / "data" / "train" / "train_finetune.json")
keys = list(train_sets)
ov = pd.DataFrame([[len(train_sets[a] & train_sets[b]) for b in keys] for a in keys], index=keys, columns=keys)
print("\nShared auxiliary (non-eval) training texts:")
print(ov.to_string())
ov.to_csv(CACHE_DIR / f"aux_overlap_{N_PAIRS}pairs.csv")

# ---------- 5. Independent re-implementation of LR (verifies arithmetic, not the i.i.d. model) ----------
def lr_components(mt, L, P, Pin, kappa):
    sq = np.sqrt(L); U, ut = P * sq, mt * sq
    fin = np.isfinite(P)
    m_in, m_out = Pin & fin, ~Pin & fin                      # missing references count as neither
    n_in, n_out = m_in.sum(0), m_out.sum(0)
    assert (n_in > 0).all() and (n_out > 0).all(), "text without IN or OUT observations"
    Uz = np.where(fin, U, 0.0)
    mu_in = np.where(m_in, Uz, 0.0).sum(0) / n_in
    mu_out = np.where(m_out, Uz, 0.0).sum(0) / n_out
    df_in, df_out = np.maximum(n_in - 1, 0), np.maximum(n_out - 1, 0)
    var_in = np.where(m_in, (Uz - mu_in) ** 2, 0.0).sum(0) / np.maximum(df_in, 1)    # model-to-model variance
    var_out = np.where(m_out, (Uz - mu_out) ** 2, 0.0).sum(0) / np.maximum(df_out, 1)
    assert df_out.sum() > 0, "no OUT degrees of freedom for pooled variance"
    pool_out = (var_out * df_out).sum() / df_out.sum()
    pool_in = (var_in * df_in).sum() / df_in.sum() if df_in.sum() > 0 else pool_out
    assert np.isfinite(pool_in) and np.isfinite(pool_out) and pool_in > 0 and pool_out > 0
    if np.isinf(kappa):                                       # infinite shrinkage: pooled variance only
        shr_in, shr_out = np.full(n_in.shape, pool_in), np.full(n_out.shape, pool_out)
    else:
        shr_in = (var_in * df_in + kappa * pool_in) / (df_in + kappa)
        shr_out = (var_out * df_out + kappa * pool_out) / (df_out + kappa)
    pred_in, pred_out = shr_in * (1 + 1 / n_in), shr_out * (1 + 1 / n_out)          # i.i.d. predictive approx.
    assert (pred_in > 0).all() and (pred_out > 0).all() and np.isfinite(pred_in).all() and np.isfinite(pred_out).all()
    lr = (-(ut - mu_in) ** 2 / (2 * pred_in) - 0.5 * np.log(pred_in)) \
         - (-(ut - mu_out) ** 2 / (2 * pred_out) - 0.5 * np.log(pred_out))
    return dict(n_in=n_in, n_out=n_out, mu_in=mu_in / sq, mu_out=mu_out / sq,
                model_sd_in=np.sqrt(shr_in), model_sd_out=np.sqrt(shr_out),
                pred_sd_in=np.sqrt(pred_in), pred_sd_out=np.sqrt(pred_out), LR=lr)

P_aug = np.vstack([P_dev, dev_fixed]); Pin_aug = np.vstack([Pin_dev, np.zeros(dev_fixed.shape, bool)])
comp = lr_components(dev_mt, dev_L, P_aug, Pin_aug, 8)
assert np.allclose(comp["LR"], sc8["LR"], rtol=0, atol=1e-8), "LR mismatch at κ=8"
compS = lr_components(dev_mt, dev_L, P_aug, Pin_aug, kappa_sel)
assert np.allclose(compS["LR"], scS["LR"], rtol=0, atol=1e-8), f"LR mismatch at κ={kappa_sel}"
print(f"\nLR arithmetic verified at κ=8 and κ={kappa_sel}. Variances: per-text model-to-model variance of "
      "sqrt(T)·mean-loss, shrunk to pooled, × (1+1/n). This is a predictive-variance approximation that assumes "
      "i.i.d. model draws; correlated references or a different model population weaken it.")

# ---------- 4. Dev non-members at the selected strict-FPR operating point (tie-aware) ----------
def operating_point(s, y):
    fpr, tpr, thr = roc_curve(y, s)
    j = np.flatnonzero(fpr < 0.01)
    j = j[np.argmax(tpr[j])]
    return thr[j], tpr[j], fpr[j]

neg = np.flatnonzero(labels == 0)
tops = {}
for name, s in (("B_ext", cands["B_ext"]), ("LR k8", cands["LR (all OUT, κ=8 fixed)"])):
    t, tpr_op, fpr_op = operating_point(s, labels)
    assert np.isclose(tpr_op, tpr1(labels, s))
    fp_mask = (s >= t) & (labels == 0)                         # roc_curve predicts member when score >= threshold
    order = neg[np.argsort(-s[neg], kind="stable")][:20]
    tops[name] = order
    first_excluded = neg[s[neg] < t][np.argmax(s[neg][s[neg] < t])]
    tab = pd.DataFrame({
        "rank": np.arange(1, 21), "idx": order, "T": dev_L[order].astype(int),
        "target loss": dev_mt[order], "mu_IN": comp["mu_in"][order], "mu_OUT": comp["mu_out"][order],
        "model sd IN": comp["model_sd_in"][order], "model sd OUT": comp["model_sd_out"][order],
        "n_IN": comp["n_in"][order], "n_OUT": comp["n_out"][order], "score": s[order],
    })
    tab["at operating point"] = np.where(fp_mask[order], "FALSE POSITIVE",
                                np.where(order == first_excluded, "highest excluded non-member", ""))
    n_ties = int(((s == t) & (labels == 0)).sum())
    print(f"\n{name}: threshold={t:.6f} | TPR={tpr_op:.4f} | FPR={fpr_op:.4f} | "
          f"false positives={int(fp_mask.sum())} | non-members tied at threshold={n_ties}")
    print(tab.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    tab.to_csv(CACHE_DIR / f"top20_nonmembers_{name.replace(' ', '_')}_{N_PAIRS}pairs.csv", index=False)
shared = set(tops["B_ext"]) & set(tops["LR k8"])
print(f"\nNon-members in both top-20 lists: {len(shared)}")

for f in [CACHE_DIR / f"aux_overlap_{N_PAIRS}pairs.csv"] + \
         [CACHE_DIR / f"top20_nonmembers_{k}_{N_PAIRS}pairs.csv" for k in ("B_ext", "LR_k8")]:
    shutil.copy(f, DRIVE / "cache")
np.savez(CACHE_DIR / f"staged4_dev_scores_{N_PAIRS}pairs.npz", **{k: v for k, v in cands.items()})
shutil.copy(CACHE_DIR / f"staged4_dev_scores_{N_PAIRS}pairs.npz", DRIVE / "cache")
print("Saved stage results and diagnostics to Drive.")

[... log truncated ...]
1916 3.1587  3.2453       0.2616        0.2128     4      8 0.4981                            
   19 1376 224       3.9302 3.8930  3.9634       0.2183        0.2206     4      8 0.4969                            
   20 1704 117       3.6633 3.6332  3.7090       0.2213        0.2008     4      8 0.4947                            

LR k8: threshold=2.647029 | TPR=0.7120 | FPR=0.0090 | false positives=9 | non-members tied at threshold=0
 rank  idx   T  target loss  mu_IN  mu_OUT  model sd IN  model sd OUT  n_IN  n_OUT  score          at operating point
    1  140  99       3.8995 3.9201  4.0022       0.2442        0.2281     4      8 8.5278              FALSE POSITIVE
    2  546  76       2.3680 2.3941  2.5447       0.3152        0.4427     4      8 5.4609              FALSE POSITIVE
    3  637 133       2.0820 2.1065  2.1672       0.2182        0.3103     4      8 4.0766              FALSE POSITIVE
    4   78  80       3.3125 3.3074  3.3778       0.2117        0.1

In [68]:
print("Missing:", [n for n in ["inA", "inB", "p2A", "p2B", "p3A", "p3B", "p4A", "p4B"] if n not in MODELS])
print("Registered pairs:", PAIRS)
print("Training chain status:", staged_proc.poll())

Missing: ['p4A', 'p4B']
Registered pairs: [('inA', 'inB'), ('p2A', 'p2B'), ('p3A', 'p3B')]
Training chain status: 0


In [71]:
# STAGED-5 — Matched comparison: pooled-aux mocks (pair 3) vs fresh-aux mocks (pair 4) vs the real dev target.
R0 = ["inA", "inB", "p2A", "p2B"]
assert all(n in MODELS for n in R0 + ["p3A", "p3B", "p4A", "p4B"])
P0 = np.stack([tm(MODELS[n]["loss"]["train"]) for n in R0])
Pin0 = np.stack([MODELS[n]["mem"]["train"] for n in R0])
F0 = np.stack([tm(cached_shadow_losses), tm(shadow2_dev)])       # never trained on dev texts
assert np.isfinite(F0).all()

ref_sets = {k: aux_set(ROOT / "data" / f"shadow_{k}" / "train_finetune.json") for k in R0}
ref_sets |= {"shadow1001": existing["shadow1001"], "shadow1002": existing["shadow1002"]}
def mean_overlap(train_set):
    return np.mean([len(train_set & v) for v in ref_sets.values()])

targets = {"p3A": "pooled", "p3B": "pooled", "p4A": "fresh", "p4B": "fresh"}
rows = []
for m, src in targets.items():
    mt = tm(MODELS[m]["loss"]["train"]); y = MODELS[m]["mem"]["train"].astype(int)
    sc, _ = scores_all_out(mt, dev_L, F0, P0, Pin0, 8)
    rows.append({"target": m, "aux source": src,
                 "mean shared aux texts w/ refs": mean_overlap(aux_set(ROOT / "data" / f"shadow_{m}" / "train_finetune.json")),
                 "B_ext TPR": tpr1(y, sc["B_ext"]), "LR κ=8 TPR": tpr1(y, sc["LR"]),
                 "B_ext AUC": roc_auc_score(y, sc["B_ext"]), "LR κ=8 AUC": roc_auc_score(y, sc["LR"])})
sc_real, _ = scores_all_out(dev_mt, dev_L, F0, P0, Pin0, 8)
rows.append({"target": "dev target (real)", "aux source": "independent",
             "mean shared aux texts w/ refs": mean_overlap(existing["TRAIN TARGET (real)"]),
             "B_ext TPR": tpr1(labels, sc_real["B_ext"]), "LR κ=8 TPR": tpr1(labels, sc_real["LR"]),
             "B_ext AUC": roc_auc_score(labels, sc_real["B_ext"]), "LR κ=8 AUC": roc_auc_score(labels, sc_real["LR"])})
t5 = pd.DataFrame(rows)
t5["LR − B_ext"] = t5["LR κ=8 TPR"] - t5["B_ext TPR"]
print("Same references (pairs 1–2 + shadows 1001/1002), same 2,000 dev texts, κ=8 fixed:")
print(t5.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
t5.to_csv(CACHE_DIR / "staged5_matched_mock.csv", index=False); shutil.copy(CACHE_DIR / "staged5_matched_mock.csv", DRIVE / "cache")

Same references (pairs 1–2 + shadows 1001/1002), same 2,000 dev texts, κ=8 fixed:
           target  aux source  mean shared aux texts w/ refs  B_ext TPR  LR κ=8 TPR  B_ext AUC  LR κ=8 AUC  LR − B_ext
              p3A      pooled                      2811.3333     0.7342      0.8483     0.9797      0.9843      0.1141
              p3B      pooled                      2799.8333     0.7750      0.8084     0.9786      0.9809      0.0334
              p4A       fresh                        77.8333     0.6813      0.7450     0.9673      0.9725      0.0637
              p4B       fresh                        76.8333     0.6064      0.6586     0.9605      0.9676      0.0522
dev target (real) independent                        97.5000     0.6290      0.6590     0.9623      0.9669      0.0300


'/content/drive/MyDrive/MIA_phase2/cache/staged5_matched_mock.csv'

In [59]:
# DIAG-NEARDUP v2 — Scored-prefix exact 8-gram matches vs the dev target's training data, with token-matched controls.
# Uses the dev target's training set, which is unavailable for validation: diagnostic only.
from numpy.lib.stride_tricks import sliding_window_view

K = 8
_rs = np.random.default_rng(12345)
POW = (_rs.integers(1, 2**62, size=K, dtype=np.int64).astype(np.uint64) | np.uint64(1))  # fixed, stable

def kgram_hashes(ids):
    a = np.asarray(ids, dtype=np.uint64)
    if a.size < K:
        return np.empty(0, dtype=np.uint64)
    return np.unique((sliding_window_view(a, K) * POW).sum(axis=1))   # uint64 wraparound: deterministic

def tok(texts):     # same truncation as scoring and training: first MAX_LENGTH tokens
    out = []
    for i in range(0, len(texts), 256):
        out += tokenizer(texts[i:i + 256], add_special_tokens=False, truncation=True,
                         max_length=MAX_LENGTH)["input_ids"]
    return out

def build_index(token_lists):
    hs = [kgram_hashes(t) for t in token_lists]
    H = np.concatenate(hs); D = np.concatenate([np.full(h.size, d) for d, h in enumerate(hs)])
    o = np.argsort(H, kind="stable")
    return H[o], D[o], len(token_lists)

def query(qh, idx):
    H, D, nd = idx
    if qh.size == 0:
        return 0.0, 0.0
    lo, hi = np.searchsorted(H, qh, "left"), np.searchsorted(H, qh, "right")
    hit = hi > lo
    if not hit.any():
        return 0.0, 0.0
    docs = np.concatenate([D[l:h] for l, h in zip(lo[hit], hi[hit])])
    return hit.mean(), np.bincount(docs, minlength=nd).max() / qh.size   # union coverage, best single text

# Corpora: target's non-eval training texts vs token-matched controls the target never saw
tgt_texts = sorted(existing["TRAIN TARGET (real)"])
tgt_tok = tok(tgt_texts)
tgt_tokens = sum(len(t) for t in tgt_tok)
ctl_pool = sorted((existing["shadow1001"] | existing["shadow1002"]) - existing["TRAIN TARGET (real)"])
ctl_pool_tok = tok(ctl_pool)
controls = []
for seed in range(5):
    order = np.random.default_rng(1000 + seed).permutation(len(ctl_pool_tok))
    picked, total = [], 0
    for j in order:
        picked.append(ctl_pool_tok[j]); total += len(ctl_pool_tok[j])
        if total >= tgt_tokens:
            break
    controls.append(build_index(picked))
print(f"Target corpus: {len(tgt_tok)} texts / {tgt_tokens} tokens | controls: 5 samples matched to ≥{tgt_tokens} tokens")
idx_tgt = build_index(tgt_tok)

q_hash = [kgram_hashes(t) for t in tok(test_items)]          # scored prefix of each dev text
res = np.array([[*query(h, idx_tgt), *np.mean([query(h, c) for c in controls], axis=0)] for h in q_hash])
cov_t, best_t, cov_c, best_c = res.T

def fp_set(s):
    t, _, _ = operating_point(s, labels)
    return np.flatnonzero((s >= t) & (labels == 0))
grp = {"B_ext false positives": fp_set(cands["B_ext"]),
       "LR κ=8 false positives": fp_set(cands["LR (all OUT, κ=8 fixed)"]),
       "top-20 non-members (B_ext)": tops["B_ext"],
       "all non-members": np.flatnonzero(labels == 0),
       "all members": np.flatnonzero(labels == 1)}
print(pd.DataFrame([{"group": g, "n": len(ix),
                     "coverage target": cov_t[ix].mean(), "coverage control": cov_c[ix].mean(),
                     "best single match target": best_t[ix].mean(), "best single match control": best_c[ix].mean(),
                     "share best>0.5 target": (best_t[ix] > 0.5).mean(), "share best>0.5 control": (best_c[ix] > 0.5).mean()}
                    for g, ix in grp.items()]).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

fp_all = sorted(set(grp["B_ext false positives"]) | set(grp["LR κ=8 false positives"]))
print(pd.DataFrame({"idx": fp_all, "T": dev_L[fp_all].astype(int), "coverage tgt": cov_t[fp_all],
                    "best tgt": best_t[fp_all], "coverage ctl": cov_c[fp_all], "best ctl": best_c[fp_all]}
                   ).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

Target corpus: 8930 texts / 1341854 tokens | controls: 5 samples matched to ≥1341854 tokens
                     group    n  coverage target  coverage control  best single match target  best single match control  share best>0.5 target  share best>0.5 control
     B_ext false positives    8            0.111             0.043                     0.106                      0.038                  0.000                   0.000
    LR κ=8 false positives    9            0.094             0.038                     0.090                      0.034                  0.000                   0.000
top-20 non-members (B_ext)   20            0.061             0.024                     0.058                      0.021                  0.000                   0.000
           all non-members 1000            0.005             0.007                     0.004                      0.006                  0.000                   0.001
               all members 1000            0.007             0.007       

In [72]:
# FREEZE-LR — Pre-registered rule met. Single validation candidate: LR, κ=8, all OUT references, 4 pairs.
import json, zipfile, io
KAPPA_FROZEN = 8
assert len(DONE_PAIRS) == 4
names_frozen = [n for p in DONE_PAIRS for n in p]

# Reproduce the dev score through the masked implementation used for validation.
comp_dev = lr_components(dev_mt, dev_L, np.vstack([P_dev, dev_fixed]),
                         np.vstack([Pin_dev, np.zeros(dev_fixed.shape, bool)]), KAPPA_FROZEN)
assert np.allclose(comp_dev["LR"], cands["LR (all OUT, κ=8 fixed)"], rtol=0, atol=1e-8)
assert np.isclose(tpr1(labels, comp_dev["LR"]), 0.712, atol=1e-9)

# Validation references: shadows 1001/1002, final target, train target (masked for its 39 training texts),
# plus every paired shadow (OUT or IN by actual membership). The masked implementation drops missing values.
assert not mem_s1["val"].any() and not mem_s2["val"].any()
val_mt = tm(val_target_losses)
val_L = np.array([len(a) for a in val_target_losses], dtype=np.float64)
val_fixed = np.stack([tm(val_shadow_losses), tm(shadow2_val), tm(val_refs["final"]),
                      np.where(in_train_ft, np.nan, tm(val_refs["train"]))])
assert int(np.isnan(val_fixed).sum()) == 39
P_val = np.stack([tm(MODELS[n]["loss"]["val"]) for n in names_frozen])
Pin_val = np.stack([MODELS[n]["mem"]["val"] for n in names_frozen])
comp_val = lr_components(val_mt, val_L, np.vstack([P_val, val_fixed]),
                         np.vstack([Pin_val, np.zeros(val_fixed.shape, bool)]), KAPPA_FROZEN)
val_LR = comp_val["LR"]

assert val_LR.shape == (2000,) and np.isfinite(val_LR).all()
assert len(val_LR) == len(val_texts)
print("Validation IN counts:", {int(k): int((comp_val['n_in'] == k).sum()) for k in np.unique(comp_val['n_in'])})
print("Validation OUT counts:", {int(k): int((comp_val['n_out'] == k).sum()) for k in np.unique(comp_val['n_out'])})
print(f"corr(val_LR, val_B5) = {np.corrcoef(val_LR, val_B5)[0, 1]:.4f}")

prediction_path = ROOT / "data" / "val" / "prediction_LR4.csv"
submission_path = ROOT / "project_phase2_val_submission_LR4_k8.zip"
np.savetxt(prediction_path, val_LR, fmt="%.17g")
for cmd in ([sys.executable, str(ROOT / "scripts" / "make_submission.py"), "--phase", "val",
             "--prediction", str(prediction_path), "--output", str(submission_path)],
            [sys.executable, str(ROOT / "scripts" / "validate_submission.py"), str(submission_path), "--phase", "val"]):
    r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    print(r.stdout, r.stderr); assert r.returncode == 0
with zipfile.ZipFile(submission_path) as zf:
    assert np.allclose(np.loadtxt(io.BytesIO(zf.read("val/prediction.csv"))), val_LR, rtol=0, atol=1e-9)

FZ = DRIVE / "frozen_LR4_k8"; FZ.mkdir(parents=True, exist_ok=True)
json.dump({"method": "Gaussian LR on sqrt(T)·mean-loss; per-text model variance shrunk to pooled (κ=8); "
                     "predictive variance × (1+1/n); IN/OUT by actual training membership",
           "kappa": KAPPA_FROZEN, "pairs": DONE_PAIRS,
           "val_fixed_out_refs": ["shadow1001", "shadow1002", "final target", "train target (39 masked)"],
           "decision_rule": "freeze LR κ=8 if at 4 pairs it is ≥ B_ext on dev and ahead on full mock set and "
                            "STAGED-5 matched comparison; submit once",
           "dev": {"LR": 0.712, "B_ext": 0.712, "B5": 0.679},
           "mock_mean": {"LR_k8": 0.8022, "B_ext": 0.7479},
           "staged5_real_dev_matched": {"B_ext": 0.629, "LR": 0.659}},
          open(FZ / "settings.json", "w"), indent=2)
np.savez(FZ / "val_scores.npz", val_LR=val_LR, n_in=comp_val["n_in"], n_out=comp_val["n_out"])
shutil.copy(submission_path, FZ); shutil.copy(submission_path, DRIVE)
print("Frozen LR candidate saved. B5 artifacts unchanged.")

Validation IN counts: {4: 1993, 5: 7}
Validation OUT counts: {6: 3, 7: 40, 8: 1957}
corr(val_LR, val_B5) = 0.9072
Created /content/Project-phase-2/project_phase2_val_submission_LR4_k8.zip with 2000 scores
 
Submission validation: PASS (2000 scores)
Score range: [-21.5441, 45.433]
 
Frozen LR candidate saved. B5 artifacts unchanged.


In [73]:
# FINAL-LR — Frozen method (LR, κ=8, 4 pairs) on the final phase. Build and save only.
fl = {k: load_split(f"final_{k}_losses.npz") for k in ("target", "shadow1", "shadow2", "train_tgt", "val_tgt")}
fin_mt = tm(fl["target"])
fin_L = np.array([len(a) for a in fl["target"]], dtype=np.float64)
for k, v in fl.items():
    assert len(v) == 2000 and all(a.shape == b.shape for a, b in zip(v, fl["target"])), k

in_tf_final = np.array([normalize_text(t) in train_ft for t in final_texts])
print("Final texts excluded per reference | shadow1001:", int(mem_s1["final"].sum()),
      "| shadow1002:", int(mem_s2["final"].sum()), "| train target:", int(in_tf_final.sum()))
fin_fixed = np.stack([np.where(mem_s1["final"], np.nan, tm(fl["shadow1"])),
                      np.where(mem_s2["final"], np.nan, tm(fl["shadow2"])),
                      np.where(in_tf_final, np.nan, tm(fl["train_tgt"])),
                      tm(fl["val_tgt"])])
P_fin = np.stack([tm(MODELS[n]["loss"]["final"]) for n in names_frozen])
Pin_fin = np.stack([MODELS[n]["mem"]["final"] for n in names_frozen])
comp_fin = lr_components(fin_mt, fin_L, np.vstack([P_fin, fin_fixed]),
                         np.vstack([Pin_fin, np.zeros(fin_fixed.shape, bool)]), KAPPA_FROZEN)
final_LR = comp_fin["LR"]
assert final_LR.shape == (2000,) and np.isfinite(final_LR).all()
print("Final IN counts:", {int(k): int((comp_fin['n_in'] == k).sum()) for k in np.unique(comp_fin['n_in'])})
print(f"Score spread  val: mean={val_LR.mean():.3f} sd={val_LR.std():.3f} | "
      f"final: mean={final_LR.mean():.3f} sd={final_LR.std():.3f}")

p = ROOT / "data" / "final" / "prediction_LR4.csv"
z = ROOT / "project_phase2_final_submission_LR4_k8.zip"
np.savetxt(p, final_LR, fmt="%.17g")
for cmd in ([sys.executable, str(ROOT / "scripts" / "make_submission.py"), "--phase", "final",
             "--prediction", str(p), "--output", str(z)],
            [sys.executable, str(ROOT / "scripts" / "validate_submission.py"), str(z), "--phase", "final"]):
    r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    print(r.stdout, r.stderr); assert r.returncode == 0
with zipfile.ZipFile(z) as zf:
    assert np.allclose(np.loadtxt(io.BytesIO(zf.read("final/prediction.csv"))), final_LR, rtol=0, atol=1e-9)
np.savez(FZ / "final_scores.npz", final_LR=final_LR, n_in=comp_fin["n_in"], n_out=comp_fin["n_out"])
shutil.copy(z, FZ); shutil.copy(z, DRIVE)
print("Final-phase LR submission built and saved. Upload only when the final phase opens.")

Final texts excluded per reference | shadow1001: 26 | shadow1002: 26 | train target: 31
Final IN counts: {4: 1977, 5: 23}
Score spread  val: mean=1.327 sd=7.401 | final: mean=1.163 sd=7.789
Created /content/Project-phase-2/project_phase2_final_submission_LR4_k8.zip with 2000 scores
 
Submission validation: PASS (2000 scores)
Score range: [-23.4796, 52.6023]
 
Final-phase LR submission built and saved. Upload only when the final phase opens.


In [74]:
# FINAL BACKUP — everything needed to resume or reproduce, copied to Drive.
for n in ["shadow_1001", "shadow_1002", "shadow_inA", "shadow_inB",
          "shadow_p2A", "shadow_p2B", "shadow_p3A", "shadow_p3B", "shadow_p4A", "shadow_p4B"]:
    for kind in ("models", "data"):
        src = ROOT / kind / n
        if src.exists():
            shutil.copytree(src, DRIVE / kind / n, dirs_exist_ok=True)
for f in (ROOT / "data").glob("*_assignment.json"):
    shutil.copy(f, DRIVE / "data")
for f in (ROOT / "data").glob("inpair*_halves.json"):
    shutil.copy(f, DRIVE / "data")
shutil.copytree(CACHE_DIR, DRIVE / "cache", dirs_exist_ok=True)
for z in ROOT.glob("project_phase2_*.zip"):
    shutil.copy(z, DRIVE)
for f in ROOT.glob("train_shadow_*.log"):
    shutil.copy(f, DRIVE / "logs")

print("Adapters on Drive:", sorted(p.parent.name for p in (DRIVE / "models").glob("*/adapter_model.safetensors")))
print("Frozen method files:", sorted(p.name for p in FZ.iterdir()))
print("ZIPs on Drive:", sorted(p.name for p in DRIVE.glob("project_phase2_*.zip")))

Adapters on Drive: ['shadow_1001', 'shadow_1002', 'shadow_inA', 'shadow_inB', 'shadow_p2A', 'shadow_p2B', 'shadow_p3A', 'shadow_p3B', 'shadow_p4A', 'shadow_p4B']
Frozen method files: ['final_scores.npz', 'project_phase2_final_submission_LR4_k8.zip', 'project_phase2_val_submission_LR4_k8.zip', 'settings.json', 'val_scores.npz']
ZIPs on Drive: ['project_phase2_final_submission_B5.zip', 'project_phase2_final_submission_B5_fix.zip', 'project_phase2_final_submission_LR4_k8.zip', 'project_phase2_val_submission.zip', 'project_phase2_val_submission_3ref.zip', 'project_phase2_val_submission_B.zip', 'project_phase2_val_submission_B4.zip', 'project_phase2_val_submission_B5.zip', 'project_phase2_val_submission_B5_gIN.zip', 'project_phase2_val_submission_LR4_k8.zip']
